# Reproducing Steinmetz et al. (2019) from dandiset 000017

**Paper.** Steinmetz NA, Zatka-Haas P, Carandini M, Harris KD (2019). Distributed coding of choice, action and engagement across the mouse brain. *Nature* 576, 266-273. [doi:10.1038/s41586-019-1787-x](https://doi.org/10.1038/s41586-019-1787-x). The claims file (`claims/steinmetz-2019.yaml`) was written from the accepted author manuscript in Europe PMC (PMC6913580) with its Methods, Extended Data legends and both supplement files.

**Dandiset.** [DANDI:000017](https://dandiarchive.org/dandiset/000017/0.240329.1926), version 0.240329.1926 (doi:10.48324/dandi.000017/0.240329.1926). The paper cites no DANDI version (its data statement names the figshare deposit), so the most recent published version is pinned; it is the only one, and the draft has the same 39 files and size.

**What the dandiset holds.** One NWB file per session, 39 sessions of 10 mice: spike times of curated units with the location of every channel, the trial table (stimulus, go cue, response, feedback, the dataset's own `included` flag for engaged trials), the wheel position and detected wheel movements, licks, pupil area and face motion energy, the passive replay events and the receptive-field mapping stimuli. It holds no histology (DiI spots).

**Method sources.** Every method is the claim's criterion, or the passage of the paper it cites; details taken from the paper are cited where they are used and in the helper module `steinmetz.py`. The paper's code (github.com/nsteinme/steinmetz-et-al-2019, commit 9a0bea8) is cited where the claims file names it. Random draws (matched-contrast subsets, NoGo movement times, shuffles, folds) use fixed seeds, one generator per use.

| Claim | Status |
|---|---|
| fig1c-easy-performance, fig1c-low-contrast-performance, fig1c-competing-performance | tested |
| fig1d-reaction-time-difficulty, ed1q-wheel-choice-decoding, early-response-fraction, passive-no-movement | tested |
| recording-counts, edtable1-region-counts, supptable2-sessions | tested |
| task-modulated-fraction, suppfig2-rate-amplitude | tested |
| fig2de-activity-order, fig2fg-miss-restricted, fig5b-miss-more-than-passive | tested |
| fig5c-prestimulus-task-vs-passive, ed6b-prestimulus-order, fig5d-go-miss-matches-task-passive, ed7f-detect-probability-engagement | tested |
| fig5f-engagement-index, fig5e-engagement-predicts-go, ed9i-engagement-beyond-arousal, ed9cf-engagement-controls, ed9k-engagement-vector-specific, suppfig3-nonresponsive-neurons | tested |
| ed7b-detect-probability-widespread | tested |
| fig4d-jpecc-vis-leads, fig4d-jpecc-no-lag-frontal-midbrain | tested |
| fig3c-variance-explained, fig3d-vision-localized, fig3e-action-global, ed4f-focality, ed5h-choice-shuffle-fpr, fig4b-choice-regions, fig4c-choice-timing, fig4f-contralateral-preference, fig4fg-opposed-activity, ed5g-ipsilateral-vision-rare, ed7c-choice-probability-restricted | blocked: the kernel regression does not fit the time limit (timed below) |
| histology-alignment-error | blocked: no DiI spot coordinates or landmarks in the dandiset |

## Data check

Counts by subject and session, the ranges and units of the variables used, and each check of the metadata against the values.

In [ ]:
import os, time, warnings, resource
import numpy as np
import pandas as pd
import scipy.stats as ss
import statsmodels.formula.api as smf
import statsmodels.api as sm_api
import matplotlib.pyplot as plt
import steinmetz as sm
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)
T0 = time.time()

In [ ]:
# Load every session once and keep only the per-session measures (steinmetz.summarize).
# Each session has its own generator for the random draws (seeded by its index).
paths = [p for p, _ in sm.asset_paths()]
print(len(paths), "NWB files in", sm.DANDISET, sm.VERSION)
R = []
for i, p in enumerate(paths):
    S = sm.Session(p, np.random.default_rng([1, i]))
    R.append(sm.summarize(S, i))
    del S
print(f"loaded in {time.time() - T0:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
meta = pd.DataFrame([r["meta"] for r in R])
meta["date"] = meta["start"].str[:10]
trials = pd.concat([r["trials"] for r in R], ignore_index=True)
units = pd.concat([r["units"] for r in R], ignore_index=True)
passive = pd.concat([r["passive"] for r in R], ignore_index=True)
print("Sessions", len(meta), "| subjects", meta.subject.nunique(), "| probes", meta.probes.sum())
print(meta[["identifier", "sex", "genotype", "age", "probes", "n_units", "n_trials"]].to_string())
print("\nSessions per subject:", meta.subject.value_counts().to_dict())
# Wheel: the 'rate' attribute of wheel_position holds the sampling interval, not a rate.
dur = meta.wheel_samples * meta.wheel_rate_attr
print("\nwheel_position 'rate' attribute:", meta.wheel_rate_attr.unique(),
      "| samples x attribute = %.0f to %.0f s (session lengths)" % (dur.min(), dur.max()),
      "| conversion", meta.wheel_conversion.unique(), "mm per unit")
for nm in ["lick", "me"]:
    print(f"{nm}: 'rate' attribute %.4g to %.4g; samples x attribute = %.0f to %.0f s" % (
          meta[f"{nm}_rate_attr"].min(), meta[f"{nm}_rate_attr"].max(),
          (meta[f"{nm}_samples"] * meta[f"{nm}_rate_attr"]).min(), (meta[f"{nm}_samples"] * meta[f"{nm}_rate_attr"]).max()))
print("Last spike time %.0f to %.0f s (the recording length the intervals must match)" % (meta.last_spike.min(), meta.last_spike.max()))
print("wheel_moves: data length / timestamps length =", (meta.moves_timestamps / meta.moves_data).unique(),
      "(timestamps hold onset and offset of each movement)")
print("Values of 'phy_annotations':", units.phy.value_counts().to_dict(),
      "(only >= 2 kept)")
# Wheel displacement from stimulus onset to the registered response, by response.
d = trials.groupby("choice").disp_resp.describe(percentiles=[0.05, 0.5, 0.95])
print("\nWheel displacement (encoder units) at response time, by response_choice "
      "(+1 Left, -1 Right, 0 NoGo):\n", d.round(1))
print("20 mm (90 d.v.a., Methods) is %.0f encoder units" % (20 / meta.wheel_conversion.iloc[0]))
# Typed movements: type 1 should be Left, 2 Right (wheel_moves comments).
ty = trials[trials.t_raw.isin(["L", "R"])]
print("\nFirst movement type of Left/Right trials (T) against response_choice:\n",
      pd.crosstab(ty.t_raw, ty.choice))
print("\nTrials by definition T (engaged only):", trials.t_choice.value_counts().to_dict(),
      "| code rule variant:", trials.code_choice.value_counts().to_dict())
agree = (trials.included == trials.rule_included)
print("Dataset 'included' against the Methods disengagement rule: agree on %.1f%% of trials; "
      "included %d, rule %d" % (100 * agree.mean(), trials.included.sum(), trials.rule_included.sum()))
print("\nContrasts in trials:", sorted(trials.cl.unique()), "| passive:", sorted(passive.pl.unique()))
print("Passive visual presentations per session: %d to %d" %
      (passive.groupby("session").size().min(), passive.groupby("session").size().max()))
# Units: region from peak_channel - 1 (peak_channel is 1-based: see the depth check below).
print("\nGood units (phy >= 2):", len(units), "| region labels:", units.region.nunique(),
      "| in the 42 regions:", units.region.isin(sm.REGIONS42).sum(),
      "| 'root':", (units.region == "root").sum())
print("Labels outside the 42 regions:",
      units.loc[~units.region.isin(sm.REGIONS42), "region"].value_counts().to_dict())
print("Trial firing rate (Hz) range %.2f to %.1f; mean spike amplitude range %.1f to %.1f (no unit given)"
      % (units.trial_rate.min(), units.trial_rate.max(), units.amp.min(), units.amp.max()))
# Duplicates: sessions with identical trial times or identical spike counts.
sig = meta.n_spikes.astype(str) + "_" + meta.n_trials.astype(str)
print("\nSessions sharing spike and trial counts:", int(sig.duplicated().sum()))
first_stim = trials.groupby("session").stim.apply(lambda x: tuple(np.round(x.to_numpy()[:5], 4)))
print("Sessions sharing their first five stimulus times:", int(first_stim.duplicated().sum()))
# Clock: decode the contralateral stimulus from the population before and after onset.
clk = pd.DataFrame([dict(session=r["meta"]["name"], **r["clock"]) for r in R])
print("\nClock check, cross-validated decoding of contralateral stimulus presence:")
print("  window -0.1 to 0 s: median %.3f (range %.3f to %.3f)" % (clk.pre.median(), clk.pre.min(), clk.pre.max()))
print("  window 0.05 to 0.15 s: median %.3f (range %.3f to %.3f)" % (clk.post.median(), clk.post.min(), clk.post.max()))
print("  chance (majority class): median %.3f; sessions with post > pre + 0.05: %d of %d"
      % (clk.chance.median(), (clk.post > clk.pre + 0.05).sum(), len(clk)))
print("  sessions where it is not established:", clk.loc[clk.post <= clk.pre + 0.05, "session"].tolist())
print("Pupil: sessions with any NaN in eye_area:", [r["meta"]["name"] for r in R
      if r["trials"]["pupil_a"].isna().any()])
if "depth_err_0" in meta:
    print("peak_channel: median |cluster depth - site depth| using row peak_channel-1: %.1f um; "
          "using row peak_channel: %.1f um" % (meta.depth_err_0.median(), meta.depth_err_1.median()))

In [ ]:
# Shared trial classes for the behavioural claims (claims fig1c-*).
def cond_class(df):
    hi, lo = np.maximum(df.cl, df.cr), np.minimum(df.cl, df.cr)
    c = np.full(len(df), "", dtype=object)
    c[(hi == 1.0) & (lo == 0)] = "high_single"
    c[(hi == 0.25) & (lo == 0)] = "low_single"
    c[((hi == 1.0) & (lo == 0.5)) | ((hi == 0.5) & (lo == 0.25))] = "competing"
    return c

trials["cls"] = cond_class(trials)
# Correct side: the higher contrast; response_choice +1 is Left, -1 Right.
trials["correct_side"] = np.where(trials.cl > trials.cr, 1, np.where(trials.cr > trials.cl, -1, 0))
trials["incorrect"] = (trials.choice != 0) & (trials.choice == -trials.correct_side)
trials["miss"] = trials.choice == 0

def perf_table(cls):
    rows = []
    for reading, sel in [("a all trials", np.ones(len(trials), bool)),
                         ("b engaged trials (included)", trials.included.to_numpy())]:
        d = trials[sel & (trials.cls == cls).to_numpy()]
        per = d.groupby("session")[["incorrect", "miss"]].mean() * 100
        rows.append(dict(reading=reading, sessions=len(per), trials=len(d),
                         incorrect_mean=per.incorrect.mean(), incorrect_sd=per.incorrect.std(),
                         miss_mean=per.miss.mean(), miss_sd=per.miss.std()))
    return pd.DataFrame(rows).round(2)

def judge_perf(tab, exp):
    (vi, ti), (vm, tm) = exp
    tab["incorrect_ok"] = (tab.incorrect_mean - vi).abs() <= ti
    tab["miss_ok"] = (tab.miss_mean - vm).abs() <= tm
    print(tab.to_string(index=False))
    print("Reproduced under a reading:", bool((tab.incorrect_ok & tab.miss_ok).any()))

## fig1c-easy-performance

> On high-contrast single-stimulus trials mice made 1.7 ± 2.5% incorrect choices and 10.1 ± 8.3% Misses (mean ± s.d. over 39 sessions of 10 mice).

High contrast is 1.0, the largest of the dataset's contrasts (0, 0.25, 0.5, 1). An incorrect choice is a `response_choice` towards the side without the stimulus; a Miss is `response_choice` 0. Expected 1.7 ± 2.5 and 10.1 ± 8.3.

In [ ]:
judge_perf(perf_table("high_single"), [(1.7, 2.5), (10.1, 8.3)])
print("By subject (reading b):")
d = trials[trials.included & (trials.cls == "high_single")]
print((d.groupby("subject")[["incorrect", "miss"]].mean() * 100).round(1).T.to_string())

**Result: reproduced.** On engaged trials (reading b) the mean over 39 sessions is 1.71% incorrect choices (s.d. 2.47) and 10.12% Misses (s.d. 8.32), against 1.7 ± 2.5 and 10.1 ± 8.3. With all trials (reading a) Misses are 23.24%, outside tolerance, so the paper's numbers are those of engaged trials.

## fig1c-low-contrast-performance

> With low-contrast single stimuli mice made 5.1 ± 6.4% incorrect choices and 29.8 ± 19.8% Misses.

Low contrast is 0.25.

In [ ]:
judge_perf(perf_table("low_single"), [(5.1, 6.4), (29.8, 19.8)])
d = trials[trials.included & (trials.cls == "low_single")]
print((d.groupby("subject")[["incorrect", "miss"]].mean() * 100).round(1).T.to_string())

**Result: reproduced.** Under both readings the values are within tolerance: 5.07% incorrect and 29.84% Misses on engaged trials (s.d. 6.43 and 19.81), against 5.1 and 29.8.

## fig1c-competing-performance

> With competing stimuli of similar but unequal contrast (high against medium, medium against low) mice made 20.0 ± 7.9% incorrect choices and 13.9 ± 11.7% Misses.

High against medium is 1.0 against 0.5; medium against low 0.5 against 0.25.

In [ ]:
judge_perf(perf_table("competing"), [(20.0, 7.9), (13.9, 11.7)])
d = trials[trials.included & (trials.cls == "competing")]
print((d.groupby("subject")[["incorrect", "miss"]].mean() * 100).round(1).T.to_string())

**Result: reproduced.** Engaged trials give 20.01% incorrect choices and 13.89% Misses (s.d. 7.90 and 11.66), against 20.0 and 13.9; all trials give 19.93% and 14.66%.

## fig1d-reaction-time-difficulty

> Reaction times were longer in the harder conditions (low-contrast single stimuli and competing stimuli of similar contrast) than for high-contrast single stimuli.

Reaction time is the detected movement onset minus stimulus onset. Trial readings: (a) all trials with a Left or Right response (`response_choice`), the reaction time being the first detected movement onset after stimulus onset; (b) the Left and Right choice trials of definition T (onset 0.125 to 0.4 s). Factor readings F1 (class and session), F2 (class and subject), F3 (contrast, competing stimulus present, session). Type II ANOVA on an ordinary least squares fit (statsmodels), a standard choice for a multi-way ANOVA.

In [ ]:
rows = []
for tr_read in ["a", "b"]:
    if tr_read == "a":
        d = trials[(trials.choice != 0) & (trials.cls != "")].copy()
        d["rt"] = d.any_onset
        d = d[d.rt.notna() & (d.rt > 0)]
    else:
        d = trials[trials.t_choice.isin(["L", "R"]) & (trials.cls != "")].copy()
        d["rt"] = d.first_move
    d["contrast"] = np.maximum(d.cl, d.cr)
    d["competing"] = (np.minimum(d.cl, d.cr) > 0).astype(int)
    means = d.groupby("cls").rt.mean()
    for f_read, formula, terms in [("F1", "rt ~ C(cls) + C(session)", ["C(cls)"]),
                                   ("F2", "rt ~ C(cls) + C(subject)", ["C(cls)"]),
                                   ("F3", "rt ~ C(contrast) + C(competing) + C(session)",
                                    ["C(contrast)", "C(competing)"])]:
        a = sm_api.stats.anova_lm(smf.ols(formula, d).fit(), typ=2)
        for t in terms:
            rows.append(dict(trials=tr_read, factors=f_read, term=t, F=a.loc[t, "F"],
                             df=a.loc[t, "df"], p=a.loc[t, "PR(>F)"], n=len(d),
                             rt_high=means["high_single"], rt_low=means["low_single"],
                             rt_comp=means["competing"]))
rt = pd.DataFrame(rows)
rt["high_shortest"] = (rt.rt_high < rt.rt_low) & (rt.rt_high < rt.rt_comp)
print(rt.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("Reproduced under a reading:", bool(((rt.p < 1e-4) & rt.high_shortest).any()))
d = trials[trials.t_choice.isin(["L", "R"]) & (trials.cls != "")]
print("\nMedian reaction time (s) by subject and class, reading b:")
print(d.groupby(["subject", "cls"]).first_move.median().unstack().round(3).to_string())

**Result: reproduced.** On the Left and Right trials of definition T (reading b), the mean reaction time is 0.206 s for high-contrast single stimuli, 0.226 s for low-contrast single stimuli and 0.221 s for competing stimuli, and the condition factor has p = 2.9e-10 (F1), 8.7e-09 (F2) and 9.6e-10 for contrast (F3), all below 1e-4. Reading a (first movement after the stimulus on all trials with a Left or Right response) gives p = 0.006 to 0.02 and the competing class shortest (0.410 s against 0.413 s), so the claim holds under reading b only.

## ed1q-wheel-choice-decoding

> Choice decoded from instantaneous wheel velocity is at chance (47.7%) 50 ms before detected movement onset and near perfect (94.6%) 50 ms after it.

Velocity is the wheel position at t minus at t - 10 ms, around the detected onset of Left and Right choice movements of definition T. Reading (a) the sign of the velocity (positive is Left, as the response check in the data check shows; zero counts 50%); (b) logistic regression on velocity, 10-fold cross-validation over trials per session (scikit-learn, default regularization, a standard choice). Accuracy is the mean over sessions.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
g = trials[trials.t_choice.isin(["L", "R"])]
rows = []
for s, d in g.groupby("session"):
    y = (d.t_choice == "L").astype(int).to_numpy()
    for o in sm.VEL_OFFSETS:
        v = d[f"vel_{o:+.2f}"].to_numpy()
        acc_a = np.mean(np.where(v == 0, 0.5, (v > 0) == y))
        cv = StratifiedKFold(10, shuffle=True, random_state=0)
        acc_b = cross_val_score(LogisticRegression(), v[:, None], y, cv=cv).mean()
        rows.append(dict(session=s, offset=o, a=acc_a * 100, b=acc_b * 100))
dec = pd.DataFrame(rows)
summ = dec.groupby("offset")[["a", "b"]].agg(["mean", "std"]).round(1)
print(summ.to_string())
res = summ.loc[[-0.05, 0.05]]
ok = {r: bool(abs(res.loc[-0.05, (r, "mean")] - 47.7) <= 4.77 and
              abs(res.loc[0.05, (r, "mean")] - 94.6) <= 9.46) for r in ["a", "b"]}
print("Within tolerance (47.7 ± 4.77 at -50 ms, 94.6 ± 9.46 at +50 ms):", ok)
print("Fraction of trials with zero velocity at -50 ms: %.3f" % (g["vel_-0.05"] == 0).mean())
fig, ax = plt.subplots(figsize=(4, 3))
for r in ["a", "b"]:
    ax.plot(summ.index * 1000, summ[(r, "mean")], label=f"reading {r}")
ax.axhline(50, color="k", lw=0.5); ax.set_xlabel("time from movement onset (ms)")
ax.set_ylabel("choice decoded (%)"); ax.legend(); plt.show()
print("Plotted range: %.1f to %.1f%%" % (summ.xs("mean", axis=1, level=1).min().min(),
                                          summ.xs("mean", axis=1, level=1).max().max()))

**Result: reproduced.** With the sign of the velocity (reading a), choice is decoded on 48.9% of trials 50 ms before movement onset and 94.8% 50 ms after, against 47.7 and 94.6. At -50 ms 92.0% of trials have zero velocity. The logistic decoder (reading b) gives 59.0% before onset, above tolerance, because with zero velocity it predicts the more frequent choice.

## early-response-fraction

> On trials that ended in a Left or Right choice, mice turned the wheel within 400 ms of stimulus onset on 64.9 ± 14.0% (s.d.) of trials, nearly always (96.6 ± 3.4%) in the direction of their final choice.

Trials whose `response_choice` is Left or Right, all trials of each session. Reading (a): the first choice-amplitude turn (types 1 and 2) after stimulus onset begins within 0.4 s, its direction its type; (b): the first movement of any kind after stimulus onset begins within 0.4 s, its direction the sign of the wheel displacement over the movement.

In [ ]:
g = trials[trials.choice != 0].copy()
resp_dir = np.where(g.choice == 1, 1, -1)       # +1 Left
rows = []
g["a_early"] = (g.typed_onset >= 0) & (g.typed_onset <= 0.4)
g["a_same"] = np.where(g.typed_type == 1, 1, -1) == resp_dir
g["b_early"] = (g.any_onset >= 0) & (g.any_onset <= 0.4)
g["b_same"] = g.any_dir.to_numpy() == resp_dir
for r in ["a", "b"]:
    per = g.groupby("session").apply(lambda d: pd.Series(dict(
        early=100 * d[f"{r}_early"].mean(),
        same=100 * d.loc[d[f"{r}_early"], f"{r}_same"].mean())))
    rows.append(dict(reading=r, early_mean=per.early.mean(), early_sd=per.early.std(),
                     same_mean=per.same.mean(), same_sd=per.same.std()))
er = pd.DataFrame(rows).round(2)
er["ok"] = ((er.early_mean - 64.9).abs() <= 14.0) & ((er.same_mean - 96.6).abs() <= 3.4)
print(er.to_string(index=False))
print("Direction sign check: in reading (b), Left responses have positive displacement on %.1f%% "
      "of early trials" % (100 * (g.loc[g.b_early & (g.choice == 1), "any_dir"] > 0).mean()))

**Result: reproduced.** Under reading a, 61.48% (s.d. 12.00) of trials with a Left or Right response have a choice turn within 0.4 s, 95.78% (s.d. 3.86) of them in the direction of the response; under reading b, 67.49% and 94.93%. Both are within tolerance of 64.9 ± 14.0 and 96.6 ± 3.4.

## passive-no-movement

> In the passive replay mice hardly ever turned the wheel; 94.1% ± 0.6% of passive trials with high-contrast stimuli had no movement.

Passive visual presentations with contrast 1.0 on either side. No movement is no detected movement overlapping the window (onset or in progress). Windows (a) -0.05 to 0.4 s, (b) 0 to 0.4 s; values (i) mean over sessions, (ii) pooled.

In [ ]:
h = passive[(passive.pl == 1) | (passive.pr == 1)]
rows = []
for w in ["a", "b"]:
    still = ~h[f"moving_{w}"]
    per = still.groupby(h.session).mean() * 100
    rows.append(dict(window=w, value="i mean over sessions", pct=per.mean(), sd=per.std(),
                     sem=per.sem(), n=len(per)))
    rows.append(dict(window=w, value="ii pooled", pct=still.mean() * 100, sd=np.nan,
                     sem=np.nan, n=len(h)))
pn = pd.DataFrame(rows).round(2)
pn["ok"] = (pn.pct - 94.1).abs() <= 0.6
print(pn.to_string(index=False))
print("Per subject, window a:", (~h.moving_a).groupby(h.subject).mean().mul(100).round(1).to_dict())

**Result: not reproduced.** 84.63% of high-contrast passive presentations have no detected movement in -0.05 to 0.4 s (mean over 39 sessions, s.e. 1.08) and 84.96% in 0 to 0.4 s; pooled values are 84.55% and 84.87%. All four readings are about 9 points below 94.1 ± 0.6. Subjects range from 78.3% to 91.5%.

## Neuron sets (definition N)

Units with `phy_annotations` >= 2 (the dataset's description: "the neurons with _phy_annotation>=2 are the ones that should be included"), trial firing rate (stimulus onset to 0.4 s, trials T) at least 0.1 Hz, at least 13 such neurons of one region in one session, regions recorded in at least two subjects. Regions are the 42 of Extended Data Table 1, from the location of each unit's peak channel. Active during the task: any of the six tests below 0.05/6. The Figs. 3 to 5 set: any of tests 1 to 4 below 0.05.

In [ ]:
# Region labels: the files use finer labels for some units (SCig, SCsg, LSr, ...); these are
# mapped to their parent among the 42 (sm.REGION_PARENT). The labels as given are a variant.
units["region_raw"] = units.region
units["region"] = units.region_raw.replace(sm.REGION_PARENT)
print("Units relabelled to a parent region:", units.region_raw.isin(list(sm.REGION_PARENT)).sum(),
      units.loc[units.region_raw.isin(list(sm.REGION_PARENT)), "region_raw"].value_counts().to_dict())

def neuron_set(reg):
    in42 = reg.isin(sm.REGIONS42)
    c = units[units.rate_ok & in42].groupby(["session", reg[units.rate_ok & in42]]).size()
    okp = set(c[c >= sm.MIN_PER_REGION].index)
    sr = pd.Series([(s, r) in okp for s, r in zip(units.session, reg)], index=units.index)
    spr = units[units.rate_ok & in42 & sr].groupby(reg[units.rate_ok & in42 & sr]).subject.nunique()
    return units.rate_ok & in42 & sr & reg.isin(set(spr[spr >= 2].index))

units["rate_ok"] = units.trial_rate >= sm.MIN_RATE
units["N_raw"] = neuron_set(units.region_raw)
print("Variant, labels as given: neurons N", int(units.N_raw.sum()))
units["rate_ok"] = units.trial_rate >= sm.MIN_RATE
units["in42"] = units.region.isin(sm.REGIONS42)
cnt = units[units.rate_ok & units.in42].groupby(["session", "region"]).size()
ok_pairs = set(cnt[cnt >= sm.MIN_PER_REGION].index)
units["sess_region_ok"] = [(s, r) in ok_pairs for s, r in zip(units.session, units.region)]
subj_per_region = units[units.rate_ok & units.in42 & units.sess_region_ok].groupby("region").subject.nunique()
regions_ok = set(subj_per_region[subj_per_region >= 2].index)
units["N"] = units.rate_ok & units.in42 & units.sess_region_ok & units.region.isin(regions_ok)
P = units[[f"p{k}" for k in range(1, 7)]].to_numpy()
units["active"] = np.nanmin(P, axis=1) < 0.05 / 6
units["fig35"] = units.N & (np.nanmin(P[:, :4], axis=1) < 0.05)
units["active_nan_variant"] = units.active   # code variant noted in the text
N = units[units.N]
print("Neurons N:", len(N), "in", N.region.nunique(), "regions,", N.session.nunique(), "sessions;",
      "active:", int(N.active.sum()), "| Figs. 3-5 set:", int(units.fig35.sum()))
print("Regions of the 42 not in N:", sorted(set(sm.REGIONS42) - set(N.region)))
nan_tests = np.isnan(P[units.N.to_numpy()]).sum(axis=0)
print("Tests returning no p-value (all values equal), among N, per test:", nan_tests.tolist())

## recording-counts

> Across 39 sessions in 10 mice with 92 probe insertions, 29,134 neurons were recorded (747 ± 38 per session, mean ± s.e.), 22,458 of them localized to one of 42 brain regions; the behaviour comprised 9,538 trials.

Readings of neurons: (a) units with `phy_annotations` >= 2; (b) those with a trial firing rate of at least 0.1 Hz. Localized: (i) in one of the 42 regions; (ii) neurons N. Trials: (a) all trials; (b) engaged trials (`included`).

In [ ]:
rows = []
for nr, sel in [("a phy>=2", np.ones(len(units), bool)), ("b phy>=2, >=0.1 Hz", units.rate_ok.to_numpy())]:
    u = units[sel]
    per = u.groupby("session").size()
    for lr, loc in [("i in 42 regions", u.in42), ("ii neurons N", u.N)]:
        rows.append(dict(neurons=nr, localized=lr, total=len(u), per_session=per.mean(),
                         per_session_se=per.sem(), localized_n=int(loc.sum())))
rc = pd.DataFrame(rows)
rc["total_ok"] = (rc.total - 29134).abs() <= 2913
rc["per_ok"] = (rc.per_session - 747).abs() <= 38
rc["loc_ok"] = (rc.localized_n - 22458).abs() <= 2246
print(rc.round(1).to_string(index=False))
print("\nSessions %d (expected 39), mice %d (10), probe insertions %d (92)" %
      (len(meta), meta.subject.nunique(), meta.probes.sum()))
print("Trials: (a) all %d, (b) engaged %d; expected 9538 ± 954" % (len(trials), trials.included.sum()))
print("Units per probe count:", meta.groupby("probes").size().to_dict())

**Result: reproduced.** The dataset has 39 sessions, 10 mice and 92 probe insertions. Units curated as single units with a trial rate of at least 0.1 Hz (reading b) number 29,283 (750.8 ± 38.0 per session, mean ± s.e.), against 29,134 and 747 ± 38; 22,434 of them are in the 42 regions (reading i) and 22,273 are neurons N (reading ii), against 22,458. The engaged trials number 9,538, the paper's count exactly. Reading a (all curated units, 33,997) is outside tolerance.

## edtable1-region-counts

> Extended Data Table 1 gives, for each of the 42 regions, the number of recordings, mice, task-responsive neurons and neurons (for example MOs 19 recordings, 9 mice, 993 of 1,534 neurons responsive; MRN 11, 7, 722 of 857).

Criterion as amended after the question in `questions/steinmetz-2019.yaml`: the neurons of the table are every unit curated as a single unit with a trial firing rate of at least 0.1 Hz whose channel lies in one of the 42 regions (the Methods' "neuron"); the rule of 13 per session decides which regions are among the 42, not the table. Active during the task as task-modulated-fraction (any of the six tests below 0.05/6). Recordings are read as (a) sessions with any neuron of the region, (b) sessions with at least 13. Mice are the subjects with any neuron of the region under both readings (their count over the sessions of reading b is printed too). Definition N for all four counts is a variant (the test of the first run). Against the table (`sm.ED_TABLE1`): recordings and mice exact, neurons within 10%, in at least 38 of the 42 regions.

In [ ]:
ref = pd.DataFrame(sm.ED_TABLE1, index=["p_recordings", "p_mice", "p_responsive", "p_neurons"]).T
NT = units[units.rate_ok & units.in42]          # the table's neurons (criterion as amended)

def et_table(d, rec_min):
    per = d.groupby(["region", "session"]).size()
    keep = per[per >= rec_min].reset_index()
    g = d.groupby("region").agg(mice=("subject", "nunique"), responsive=("active", "sum"),
                                neurons=("unit", "size"))
    g["recordings"] = keep.groupby("region").session.nunique()
    g["mice_rec"] = keep.merge(d[["session", "subject"]].drop_duplicates()).groupby("region").subject.nunique()
    et = ref.join(g, how="left").fillna(0)
    et["rec_ok"] = et.recordings == et.p_recordings
    et["mice_ok"] = et.mice == et.p_mice
    et["neur_ok"] = (et.neurons - et.p_neurons).abs() <= 0.1 * et.p_neurons
    et["resp_ok"] = (et.responsive - et.p_responsive).abs() <= 0.1 * et.p_responsive
    et["all_ok"] = et.rec_ok & et.mice_ok & et.neur_ok
    return et

ets = {"a (any neuron)": et_table(NT, 1), "b (>= 13 neurons)": et_table(NT, sm.MIN_PER_REGION),
       "variant, neurons N": et_table(N, 1)}
rows = []
for k, et in ets.items():
    rows.append(dict(reading=k, all_three=int(et.all_ok.sum()), rec_exact=int(et.rec_ok.sum()),
                     mice_exact=int(et.mice_ok.sum()), neurons_10pc=int(et.neur_ok.sum()),
                     responsive_10pc=int(et.resp_ok.sum()),
                     mice_exact_over_reading_sessions=int((et.mice_rec == et.p_mice).sum()),
                     neurons=int(et.neurons.sum()), responsive=int(et.responsive.sum()),
                     differ=", ".join(et.index[~et.all_ok])))
etr = pd.DataFrame(rows)
etr["outcome"] = np.where(etr.all_three >= 38, "reproduced", np.where(etr.all_three >= 30, "partially", "not"))
print(etr.drop(columns="differ").to_string(index=False))
for _, r in etr.iterrows():
    print("Regions that differ, %s: %s" % (r.reading, r.differ))
print("Table totals: neurons 22458, responsive 13467")
et = ets["a (any neuron)"].copy()
et["recordings_b"] = ets["b (>= 13 neurons)"].recordings
cols = ["p_recordings", "recordings", "recordings_b", "p_mice", "mice", "p_responsive", "responsive",
        "p_neurons", "neurons"]
print("\nPer region, the table (p_) against reading a (recordings_b: reading b):")
print(et[cols].astype(int).to_string())

**Result: reproduced.** Counting every neuron of the 42 regions (curated, at least 0.1 Hz), as the amended criterion does, recordings read as sessions with any neuron of the region (reading a) match exactly in 41 of 42 regions, mice in 42 and neurons within 10% in 38; all three match in 38 of 42 regions, the number needed. The regions that differ are ACA, MOs, VPL and ZI (MOs has 18 recordings against 19, and 1,352 neurons against 1,534). The neurons number 22,434 and the active ones 13,532, against 22,458 and 13,467 in the table. Reading b (sessions with at least 13 neurons) matches in 26 regions, as does definition N (the test of the first run, 26).

## supptable2-sessions

> The 39 sessions are those of Supplementary Table 2, each with its subject, date, age and recorded areas (for example Cori 2016-12-14, 11 weeks, ACA, CA3, DG, LS, MOs, SUB, VISp), from the 10 subjects of Supplementary Table 1 with their sex and genotype.

Sessions matched by subject and date. Criterion as amended after the question on edtable1-region-counts: the areas of a session are the regions (of the 42) of its neurons, a neuron being a unit that passes curation and the 0.1 Hz trial rate (Methods, Data analysis: "Units passing these criteria were considered to reflect the spiking activity of a neuron"), with the readings of edtable1-region-counts: (a) a region with any neuron in the session, (b) a region with at least 13. The regions of neurons N and of all curated units are variants. Sex and genotype from the file's subject metadata; a genotype written another way (Ai95 is the RCL-GCaMP6f reporter line; Vglut1-Cre for Vglut1-IRES2-Cre-D) is the same genotype and is listed.

In [ ]:
ref2 = pd.DataFrame(sm.SUPP_T2, columns=["subject", "date", "age_weeks", "areas"])
ref2["key"] = ref2.subject + "_" + ref2.date
meta["key"] = meta.subject + "_" + meta.date
m2 = ref2.merge(meta[["key", "identifier", "sex", "genotype", "age"]], on="key", how="left")
print("Sessions of the table found in the dataset: %d of %d; dataset sessions not in the table: %s"
      % (m2.identifier.notna().sum(), len(ref2), sorted(set(meta.key) - set(ref2.key))))
areasN = N.groupby("session").region.apply(lambda x: set(x))
areas13 = units[units.in42 & units.rate_ok & units.sess_region_ok].groupby("session").region.apply(lambda x: set(x))
areas42 = units[units.in42].groupby("session").region.apply(lambda x: set(x))
areasNeur = units[units.in42 & units.rate_ok].groupby("session").region.apply(lambda x: set(x))
m2["areas_N"] = m2.identifier.map(areasN)
m2["areas_42"] = m2.identifier.map(areas42)
m2["ref_set"] = m2.areas.str.split(", ").apply(set)
m2["match_N"] = m2.ref_set == m2.areas_N
m2["match_42"] = m2.ref_set == m2.areas_42
m2["match_neur"] = m2.ref_set == m2.identifier.map(areasNeur)
m2["match_13"] = m2.ref_set == m2.identifier.map(areas13)
m2["age_days"] = m2.age.str.extract(r"(\d+)").astype(float)[0]
m2["age_weeks_data"] = (m2.age_days / 7).round(1)
m2["areas_neur"] = m2.identifier.map(areasNeur)
m2["only_paper"] = [sorted(a - (b or set())) for a, b in zip(m2.ref_set, m2.areas_neur)]
m2["only_data"] = [sorted((b or set()) - a) for a, b in zip(m2.ref_set, m2.areas_neur)]
print(m2[["key", "age_weeks", "age_weeks_data", "match_neur", "match_13", "match_N", "match_42",
          "only_paper", "only_data"]].to_string())
print("(only_paper, only_data: areas of the table missing from reading a, and areas of reading a not in the table)")
print("\nArea sets matching: reading a (any neuron) %d of 39; reading b (>= 13 neurons) %d; "
      "variants: neurons N %d, all curated units %d" % (m2.match_neur.sum(), m2.match_13.sum(),
      m2.match_N.sum(), m2.match_42.sum()))
areasRaw = units[units.region_raw.isin(sm.REGIONS42) & units.rate_ok].groupby("session").region_raw.apply(lambda x: set(x))
print("Variant, region labels as given (no parent mapping): neurons' areas match in %d of 39 sessions"
      % (m2.ref_set == m2.identifier.map(areasRaw)).sum())
print("Ages match in %d of 39 sessions" % (m2.age_weeks == m2.age_weeks_data.round()).sum())
sx = meta.groupby("subject")[["sex", "genotype"]].first()
sx["paper_sex"] = [sm.SUPP_T1[s][0] for s in sx.index]
sx["paper_genotype"] = [sm.SUPP_T1[s][1] for s in sx.index]
print(sx.to_string())
print("Sex matches in %d of 10 subjects" % (sx.sex == sx.paper_sex).sum())

**Result: reproduced.** All 39 sessions of the table are in the dataset, with matching ages (39 of 39) and sexes (10 of 10 subjects); the genotypes are the same lines under other names. Under reading a of the amended criterion (a region with any neuron in the session) the areas match the table in 38 of 39 sessions, Muller 2017-01-08 lacking MOs; under reading b (a region with at least 13 neurons) they match in 24, as with neurons N. The match rests on mapping finer region labels to their parent; with the labels as given, 25 of 39 match.

## task-modulated-fraction

> 60.0% of neurons (13,466) were significantly modulated during the task; of these 74.3% consistently increased and 20.2% consistently decreased their activity.

Neurons N; active is any of the six tests below 0.05/6. Direction is the sign of the code's effect size for each test against baseline (1, 2, 3, 5, 6). Reading (a): all tests against baseline with p < 0.05/6 have the same sign; (b): all tests against baseline with p < 0.05 have the same sign. An active neuron with no significant test against baseline (only test 4) is neither.

In [ ]:
def direction(df, alpha):
    Pb = df[["p1", "p2", "p3", "p5", "p6"]].to_numpy()
    Eb = df[["e1", "e2", "e3", "e5", "e6"]].to_numpy()
    sig = Pb < alpha
    pos = (sig & (Eb > 0)).sum(axis=1)
    neg = (sig & (Eb < 0)).sum(axis=1)
    return (pos > 0) & (neg == 0), (neg > 0) & (pos == 0)

act = N[N.active]
rows = []
for r, alpha in [("a", 0.05 / 6), ("b", 0.05)]:
    inc, dec = direction(act, alpha)
    rows.append(dict(reading=r, active=len(act), frac_active=100 * len(act) / len(N),
                     increase=100 * inc.mean(), decrease=100 * dec.mean()))
tm = pd.DataFrame(rows)
tm["ok"] = ((tm.active - 13466).abs() <= 1347) & ((tm.frac_active - 60.0).abs() <= 6.0) & \
    ((tm.increase - 74.3).abs() <= 7.4) & ((tm.decrease - 20.2).abs() <= 2.0)
print(tm.round(2).to_string(index=False))
print("Active fraction by subject (%):", (N.groupby("subject").active.mean() * 100).round(1).to_dict())
print("Variant, labels as given (N_raw): active %d of %d" % (units[units.N_raw].active.sum(), units.N_raw.sum()))
print("Per test, fraction of N below 0.05/6:", (N[[f"p{k}" for k in range(1, 7)]] < 0.05 / 6).mean().round(3).to_dict())

**Result: reproduced.** 13,439 of 22,273 neurons N (60.34%) are active during the task; under reading a 74.95% of them consistently increase and 20.89% consistently decrease, under reading b 71.60% and 19.44%. Expected 13,466, 60.0%, 74.3% and 20.2%; both readings are within tolerance.

## suppfig2-rate-amplitude

> Statistically responsive neurons have higher baseline firing rates than other neurons (d' = 0.35 ± 0.04), while their spike amplitudes differ much less (d' = 0.10 ± 0.02).

Neurons N. Baseline is the mean rate -0.2 to 0 s before stimulus onset on trials T; amplitude is the mean of the file's `spike_amps` over the unit's spikes. d' = (mean of significant - mean of others) / sqrt((var1 + var2) / 2). Reading (a): significant on any of the six tests (p < 0.05/6) against the rest, ± a bootstrap s.e. over neurons (1,000 resamples); (b) the six per-test d' values (significant on test k at 0.05/6 against the rest), mean ± s.e.

In [ ]:
rng = np.random.default_rng(20)
def dprime(x, sig):
    a, b = x[sig], x[~sig]
    return (a.mean() - b.mean()) / np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
X = {"rate": N.baseline_rate.to_numpy(), "amp": N.amp.to_numpy()}
sig_any = N.active.to_numpy()
rows = []
for q, x in X.items():
    ok = np.isfinite(x)
    d0 = dprime(x[ok], sig_any[ok])
    boots = []
    idx = np.where(ok)[0]
    for _ in range(1000):
        b = rng.choice(idx, len(idx))
        boots.append(dprime(x[b], sig_any[b]))
    per = [dprime(x[ok], (N[f"p{k}"].to_numpy() < 0.05 / 6)[ok]) for k in range(1, 7)]
    rows.append(dict(quantity=q, a_dprime=d0, a_se=np.std(boots), b_mean=np.mean(per),
                     b_se=np.std(per, ddof=1) / np.sqrt(6), per_test=np.round(per, 3).tolist()))
sf = pd.DataFrame(rows)
print(sf.round(3).to_string(index=False))
okA = abs(sf.a_dprime[0] - 0.35) <= 0.04 and abs(sf.a_dprime[1] - 0.10) <= 0.02
okB = abs(sf.b_mean[0] - 0.35) <= 0.04 and abs(sf.b_mean[1] - 0.10) <= 0.02
print("Within tolerance: reading a", okA, "| reading b", okB)
print("Baseline rate range %.2f to %.1f Hz; amplitude range %.1f to %.1f" % (
      np.nanmin(X["rate"]), np.nanmax(X["rate"]), np.nanmin(X["amp"]), np.nanmax(X["amp"])))
print("Same with log10(rate + 1):", round(dprime(np.log10(X["rate"] + 1), sig_any), 3))

**Result: partially reproduced.** Responsive neurons have higher baseline rates: d' = 0.392 ± 0.012 (reading a, outside 0.35 ± 0.04 by 0.002) and 0.316 ± 0.013 (reading b, within tolerance). Amplitudes differ less, but d' is 0.050 ± 0.014 (a) and 0.060 ± 0.028 (b), against 0.10 ± 0.02, outside tolerance under both readings.

In [ ]:
# Per-neuron arrays in the order of `units`: normalized traces (Fig. 2) and z-scores (R).
def stack(get, width):
    out = []
    for r in R:
        n = len(r["units"])
        a = get(r)
        out.append(np.full((n, width), np.nan, np.float32) if a is None else np.asarray(a, np.float32).reshape(n, width))
    return np.concatenate(out)

nb = int(round((sm.STIM_WIN[1] - sm.STIM_WIN[0]) / sm.BIN))
t_axis = sm.STIM_WIN[0] + sm.BIN * (np.arange(nb) + 0.5)
TR = {k: stack(lambda r, k=k: r["traces"].get(k), nb) for k in
      ["contraGo", "ipsiGo", "contraMiss", "ipsiMiss", "passive"]}
Z = {}
for k in ["sGo", "sNg", "ipsiMiss", "mNg", "mPa", "sPa"]:
    for j, zz in enumerate(["Z1", "Z2"]):
        Z[(k, zz)] = stack(lambda r, k=k, j=j: r["z"][k][j], 1)[:, 0]
nsets = pd.DataFrame([dict(session=r["meta"]["name"], **{k: (len(v) if hasattr(v, "__len__") else v)
                      for k, v in r["sets"].items()}) for r in R])
print(nsets.describe().round(1).to_string())

## fig2de-activity-order

> When the mouse chose a contralateral stimulus, activity arose first in visual cortex and superficial superior colliculus and then spread to most regions; when it chose an ipsilateral stimulus, VIS and SCs were among the last areas to respond.

Correct contralateral trials: Right choice (T) on a right stimulus alone, matched in contrast with the contralateral Miss and passive trials of the same single-stimulus type (definition M applied to these three groups). Correct ipsilateral: Left choice on a left stimulus alone. Per region the mean normalized rate over responsive neurons; responsive readings (a) the sliding ranksum display selection, (b) active during the task, (c) the Figs. 3 to 5 set. Latency L1: first time in 0 to 0.4 s at which the trace reaches half its maximum; L2: the time of the maximum. Regions with at least 5 responsive neurons (a standard choice, as the Fig. 3d legend's threshold).

In [ ]:
win = (t_axis >= 0) & (t_axis <= 0.4)
def region_latency(mask, key):
    rows = []
    for reg in sm.REGIONS42:
        m = mask & (units.region == reg).to_numpy()
        tr_ = TR[key][m]
        tr_ = tr_[np.isfinite(tr_).all(axis=1)]
        if len(tr_) < 5:
            continue
        y = tr_.mean(axis=0)[win]
        tt = t_axis[win]
        mx = y.max()
        l1 = tt[np.argmax(y >= mx / 2)] if mx > 0 else np.nan
        rows.append(dict(region=reg, n=len(tr_), peak=mx, L1=l1, L2=tt[np.argmax(y)]))
    return pd.DataFrame(rows).set_index("region")

masks = {"a": (units.N & units.slide_resp).to_numpy(), "b": (units.N & units.active).to_numpy(),
         "c": units.fig35.to_numpy()}
res = []
for rd, mk in masks.items():
    for lat in ["L1", "L2"]:
        c = region_latency(mk, "contraGo")
        i = region_latency(mk, "ipsiGo")
        rc_ = c[lat].rank(method="average", pct=True)
        ri_ = i[lat].rank(method="average", pct=True)
        vis_c = c.loc[c.index.isin(sm.VIS), lat].mean()
        vis_rank_c = (c[lat] < vis_c).mean() + 0.5 * (c[lat] == vis_c).mean()
        vis_i = i.loc[i.index.isin(sm.VIS), lat].mean()
        vis_rank_i = (i[lat] < vis_i).mean()
        row = dict(responsive=rd, latency=lat, regions_contra=len(c), regions_ipsi=len(i),
                   VISp_contra_pct=rc_.get("VISp"), VISmean_contra_pct=vis_rank_c,
                   SCs_contra_pct=rc_.get("SCs"), VISp_ipsi_pct=ri_.get("VISp"),
                   VISmean_ipsi_pct=vis_rank_i, SCs_ipsi_pct=ri_.get("SCs"),
                   VISp_contra_ms=1000 * c[lat].get("VISp", np.nan),
                   SCs_contra_ms=1000 * c[lat].get("SCs", np.nan),
                   median_contra_ms=1000 * c[lat].median(), median_ipsi_ms=1000 * i[lat].median())
        row["contra_ok"] = bool(row["VISp_contra_pct"] <= 0.25 and row["VISmean_contra_pct"] <= 0.25
                                and row["SCs_contra_pct"] <= 0.25)
        row["ipsi_ok"] = bool(row["VISp_ipsi_pct"] > 2 / 3 and row["SCs_ipsi_pct"] > 2 / 3)
        res.append(row)
        if rd == "a" and lat == "L1":
            print("Reading a, L1: earliest regions on contralateral trials:",
                  c[lat].sort_values().head(8).mul(1000).round(0).to_dict())
            print("Reading a, L1: latest regions on ipsilateral trials:",
                  i[lat].sort_values().tail(8).mul(1000).round(0).to_dict())
fo = pd.DataFrame(res)
print(fo.round(3).to_string(index=False))
print("Percentiles are the region's rank among regions (0 earliest, 1 latest). Reproduced under a "
      "reading:", bool((fo.contra_ok & fo.ipsi_ok).any()))
c = region_latency(masks["a"], "contraGo")
print("Peak normalized rates, contralateral, reading a: %.2f to %.2f" % (c.peak.min(), c.peak.max()))

**Result: reproduced.** With the half-maximum latency (L1), under each responsive reading, VISp is the first region on contralateral choice trials (77.5 ms; percentile 0.024), SCs is at the 0.061 to 0.083 percentile and the VIS mean at 0.195 to 0.214; on ipsilateral choice trials VISp and SCs are at the 0.70 to 0.95 percentiles, in the latest third. With the peak time (L2) SCs is not early (0.42 to 0.56), so the claim holds under L1 only.

## fig2fg-miss-restricted

> On Miss trials with a contralateral stimulus, activity appeared in a visual pathway of visual areas, basal ganglia and several midbrain structures but did not spread to the whole brain; on Miss trials with an ipsilateral stimulus there was essentially no activity.

Fraction of neurons N per region with z > 4 (definition R; readings Z1, Z2) on contrast-matched correct contralateral Go trials, contralateral Miss trials (both single-stimulus, as for fig2de) and on ipsilateral Miss trials. Sessions with fewer than 10 matched trials per group are left out of the matched conditions. A neuron with no spikes in any baseline window has a baseline s.d. of 0 and an infinite z; it counts as significant when its post-stimulus rate is above the baseline mean (division by zero, as the formula evaluates it) or is left out of the significant ones; the paper does not say, and both are run (`zero_sd` inf and finite). The fraction of such neurons is printed.

In [ ]:
keep_s = set(nsets.session[nsets.sGo >= 10])
print("Sessions with at least 10 matched single-stimulus trials per group: %d of 39" % len(keep_s))
def sigz(z, mode):
    """z > 4; mode 'inf' counts an infinite z (zero baseline s.d.), 'finite' does not."""
    return (z > 4) if mode == "inf" else (np.isfinite(z) & (z > 4))

rows = []
for zz, mode in [(a, b) for a in ["Z1", "Z2"] for b in ["inf", "finite"]]:
    d = pd.DataFrame(dict(region=units.region, session=units.session, N=units.N,
                          go=sigz(Z[("sGo", zz)], mode), miss=sigz(Z[("sNg", zz)], mode),
                          imiss=sigz(Z[("ipsiMiss", zz)], mode),
                          has_m=~np.isnan(Z[("sGo", zz)]) & ~np.isnan(Z[("sNg", zz)]),
                          has_i=~np.isnan(Z[("ipsiMiss", zz)]), inf_go=np.isinf(Z[("sGo", zz)])))
    d = d[d.N]
    dm = d[d.has_m & d.session.isin(keep_s)]
    fr = dm.groupby("region")[["go", "miss"]].mean()
    vp = fr.index.isin(sm.VISUAL_PATHWAY)
    di = d[d.has_i]
    rows.append(dict(z=zz, zero_sd=mode, miss_vispath=fr.miss[vp].mean(), miss_other=fr.miss[~vp].mean(),
                     go_other=fr.go[~vp].mean(), go_vispath=fr.go[vp].mean(),
                     contra_miss_pooled=dm.miss.mean(), ipsi_miss_pooled=di.imiss.mean(),
                     n_neurons_matched=len(dm), n_neurons_ipsi=len(di),
                     frac_go_infinite=dm.inf_go.mean()))
    if zz == "Z1" and mode == "finite":
        print("Z1, finite z only: % of neurons with z > 4 per region")
        print((fr * 100).round(1).T.to_string())
ff = pd.DataFrame(rows)
ff["i"] = ff.miss_vispath > ff.miss_other
ff["ii"] = ff.go_other >= 2 * ff.miss_other
ff["iii"] = ff.ipsi_miss_pooled < ff.contra_miss_pooled / 2
print(ff.round(4).to_string(index=False))
print("Reproduced under a reading:", bool((ff.i & ff.ii & ff.iii).any()))

**Result: partially reproduced.** No reading meets all three parts. Counting only finite z (Z1), (i) the visual pathway has more activated neurons on contralateral Miss trials than other regions (3.39% against 1.29%) and (ii) other regions have over twice as many on Go as on Miss trials (6.75% against 1.29%), but (iii) ipsilateral Misses activate 1.19% against 2.37% for contralateral Misses, just above half. With Z2, (ii) and (iii) hold and (i) does not. Counting infinite z, 21.6% of neurons have an infinite z on Go trials and (ii) fails. Only 19 of 39 sessions have 10 matched single-stimulus trials per group.

## fig5b-miss-more-than-passive

> More neurons were significantly activated by contralateral visual stimuli on Miss trials in the task than on passive trials with the same contrasts, and passive responses were weaker than those on Miss trials.

Definition M as in the Methods: trials with contralateral contrast > 0, Go (Left or Right), NoGo and passive groups matched per contrast level. Sessions kept when (reading 1) each group has at least 10 trials, or (reading 2) the three groups together have at least 10. Fraction of neurons with z > 4 on Miss (`mNg`) and passive (`mPa`) trials; neurons (a) the Figs. 3 to 5 set, (b) neurons N.

In [ ]:
print("Sessions kept: reading 1 (>= 10 per group) %d; reading 2 (>= 10 over the three groups) %d"
      % ((nsets.mGo >= 10).sum(), (3 * nsets.mGo >= 10).sum()))
keep_m = set(nsets.session[nsets.mGo >= 10])
rows = []
for zz, mode in [(a, b) for a in ["Z1", "Z2"] for b in ["inf", "finite"]]:
    for nr, msk in [("a Figs 3-5 set", units.fig35), ("b N", units.N)]:
        d = pd.DataFrame(dict(region=units.region, session=units.session,
                              miss=sigz(Z[("mNg", zz)], mode), pas=sigz(Z[("mPa", zz)], mode),
                              ok=~np.isnan(Z[("mNg", zz)]) & ~np.isnan(Z[("mPa", zz)]),
                              inf=np.isinf(Z[("mNg", zz)]) | np.isinf(Z[("mPa", zz)])))
        d = d[msk.to_numpy() & d.ok & d.session.isin(keep_m)]
        fr = d.groupby("region")[["miss", "pas"]].mean()
        ex = fr.miss - fr.pas
        rows.append(dict(z=zz, zero_sd=mode, neurons=nr, n=len(d), miss_pooled=d.miss.mean(),
                         passive_pooled=d.pas.mean(), regions=len(fr),
                         regions_excess_pos=(ex > 0).sum(), regions_excess_neg=(ex < 0).sum(),
                         frac_infinite=d.inf.mean()))
fb = pd.DataFrame(rows)
fb["ok"] = (fb.miss_pooled > fb.passive_pooled) & (fb.regions_excess_pos > fb.regions / 2) & (len(keep_m) == 34)
print(fb.round(4).to_string(index=False))
print("Reproduced under a reading:", bool(fb.ok.any()))

**Result: partially reproduced.** Contralateral stimuli activate more neurons on matched Miss trials than on passive trials: with Z1, pooled fractions 18.14% against 14.71% (Figs. 3 to 5 set, infinite z counted) or 4.82% against 3.24% (finite z), and Miss exceeds passive in 26 to 30 of 42 regions. With Z2 and finite z, fractions are near zero and the excess is positive in 3 regions. The matched-contrast selection keeps 36 sessions (38 if the 10 trials are counted over the three groups), not the paper's 34, which the criterion requires.

## fig5c-prestimulus-task-vs-passive

> Before the stimulus, neocortex and sensory thalamus were less active in the task than in the passive context, while basal ganglia and other subcortical choice areas were more active; VISp was lower and CP higher on Miss trials than on passive trials.

Per region of neurons N, the nested ANOVA of the Methods on each neuron's mean rate 0.25 to 0.05 s before stimulus onset, log10(x + 1), trials with a wheel movement in that window dropped. Task trials are trials T; passive readings (a) all passive replay events (visual, beeps, clicks, noise), (b) passive visual presentations. Factor readings (a) condition, session and neuron nested in session: with one observation per neuron and condition, the condition F of this model equals the square of a paired t across neurons, which is how it is computed; (b) condition, session and subject (sequential sums of squares, condition first). A region differs when p < 0.0012. VISp and CP: mean pre-stimulus rate over neurons on contrast-matched (M) Miss against passive trials, windows -0.2 to 0 s and -0.25 to -0.05 s.

In [ ]:
uidx = np.split(np.arange(len(units)), np.cumsum([len(r["units"]) for r in R])[:-1])
sess_names = [r["meta"]["name"] for r in R]
def prestim_means(win, pas_read):
    """Per neuron: mean pre-stimulus rate on task trials T and on passive events, trials with
    movement in the window dropped."""
    task = np.full(len(units), np.nan)
    pas = np.full(len(units), np.nan)
    for r, ix in zip(R, uidx):
        tr = r["trials"]
        okt = tr.t_choice.isin(["L", "R", "N"]).to_numpy() & ~tr[f"moving_{win}"].to_numpy()
        pre = r["pre"]
        okp = ~pre[f"pas_moving_{win}"]
        if pas_read == "b":
            okp &= pre["pas_kind"] == "visual"
        task[ix] = pre[f"task_{win}"][:, okt].mean(axis=1)
        pas[ix] = pre[f"pas_{win}"][:, okp].mean(axis=1)
    return task, pas

anova_rows = []
for pas_read in ["a", "b"]:
    task, pas = prestim_means("b", pas_read)
    lt, lp = np.log10(task + 1), np.log10(pas + 1)
    for reg in sm.REGIONS42:
        m = (units.N & (units.region == reg)).to_numpy()
        d = lt[m] - lp[m]
        n = m.sum()
        t = d.mean() / (d.std(ddof=1) / np.sqrt(n))
        pa = ss.f.sf(t ** 2, 1, n - 1)
        long = pd.DataFrame(dict(y=np.concatenate([lt[m], lp[m]]), cond=np.repeat([1, 0], n),
                                 session=np.tile(units.session[m].to_numpy(), 2),
                                 subject=np.tile(units.subject[m].to_numpy(), 2)))
        fit = smf.ols("y ~ C(cond) + C(session) + C(subject)", long).fit()
        pb = sm_api.stats.anova_lm(fit, typ=1).loc["C(cond)", "PR(>F)"]
        anova_rows.append(dict(passive=pas_read, region=reg, n=n, diff_log=d.mean(),
                               task_hz=task[m].mean(), passive_hz=pas[m].mean(), p_a=pa, p_b=pb))
an = pd.DataFrame(anova_rows)
an["neocortex"] = an.region.isin(sm.NEOCORTEX)
an["vis_thal"] = an.region.isin(sm.VIS_THALAMUS)
print(an[an.passive == "a"].round(4).to_string(index=False))

In [ ]:
# VISp and CP on matched Miss against passive trials.
def matched_means(win, key):
    out = np.full(len(units), np.nan)
    for r, ix, s in zip(R, uidx, sess_names):
        if s not in keep_m:
            continue
        idx = r["sets"][key]
        arr = r["pre"][f"pas_{win}"] if key == "mPa" else r["pre"][f"task_{win}"]
        out[ix] = arr[:, idx].mean(axis=1)
    return out
vc = []
for win in ["a", "b"]:
    mi, pa_ = matched_means(win, "mNg"), matched_means(win, "mPa")
    for reg in ["VISp", "CP"]:
        m = (units.N & (units.region == reg)).to_numpy() & np.isfinite(mi)
        dd = mi[m] - pa_[m]
        vc.append(dict(window=win, region=reg, n=m.sum(), miss_hz=mi[m].mean(), passive_hz=pa_[m].mean(),
                       diff=dd.mean(), paired_t_p=ss.ttest_rel(mi[m], pa_[m]).pvalue))
vc = pd.DataFrame(vc)
print(vc.round(4).to_string(index=False))
rows = []
for pas_read in ["a", "b"]:
    for fac in ["a", "b"]:
        d = an[an.passive == pas_read]
        dif = d[d[f"p_{fac}"] < 0.0012]
        neo = dif[dif.neocortex]
        non = dif[~dif.neocortex & ~dif.vis_thal]
        for win in ["a", "b"]:
            v = vc[vc.window == win].set_index("region")
            rows.append(dict(passive=pas_read, factors=fac, window=win, differing=len(dif),
                             neo_differing=len(neo), neo_lower=(neo.diff_log < 0).sum(),
                             non_differing=len(non), non_higher=(non.diff_log > 0).sum(),
                             VISp_lower=v.loc["VISp", "diff"] < 0, CP_higher=v.loc["CP", "diff"] > 0))
fc = pd.DataFrame(rows)
fc["ok"] = (fc.neo_lower == fc.neo_differing) & (fc.non_higher == fc.non_differing) & fc.VISp_lower & fc.CP_higher
print(fc.to_string(index=False))
for pas_read in ["a", "b"]:
    d = an[(an.passive == pas_read) & (an.p_a < 0.0012)]
    print(f"Passive reading {pas_read}, factors a: lower in task:", d[d.diff_log < 0].region.tolist(),
          "| higher:", d[d.diff_log > 0].region.tolist())
print("Reproduced under a reading:", bool(fc.ok.any()))

**Result: reproduced.** Under the Methods' ANOVA (factors a) with all passive events, 23 regions differ: all 9 differing neocortical regions are lower in the task, and all 11 differing non-neocortical regions outside visual thalamus are higher. VISp is lower on matched Miss than passive trials (1.64 against 2.39 Hz) and CP higher (3.10 against 2.77 Hz). Factors b give 14 differing regions with the same pattern; with passive visual presentations and factors a, ILA (neocortex) is higher in the task, so that reading fails.

## ed6b-prestimulus-order

> Pre-stimulus firing in midbrain, basal ganglia and hippocampal regions is ordered Passive < Miss < Go, while in neocortical areas it is higher in Passive than in Miss and Go.

Per region, the mean over neurons of the Figs. 3 to 5 set of the mean pre-stimulus rate on contrast-matched (M) Go, Miss and passive trials, sessions kept for M; windows (a) -0.2 to 0 s, (b) -0.25 to -0.05 s.

In [ ]:
rows = []
order_tabs = {}
for win in ["a", "b"]:
    go_, mi, pa_ = matched_means(win, "mGo"), matched_means(win, "mNg"), matched_means(win, "mPa")
    d = pd.DataFrame(dict(region=units.region, go=go_, miss=mi, pas=pa_))[units.fig35.to_numpy() & np.isfinite(go_)]
    g = d.groupby("region")[["pas", "miss", "go"]].mean()
    g["ordered"] = (g.pas < g.miss) & (g.miss < g.go)
    g["pas_top"] = (g.pas > g.miss) & (g.pas > g.go)
    order_tabs[win] = g
    grp = {"midbrain": sm.ED6B_MIDBRAIN, "basal ganglia": sm.ED6B_BASAL_GANGLIA,
           "hippocampus": sm.ED6B_HIPPOCAMPUS}
    sub = g[g.index.isin(sum(grp.values(), []))]
    neo = g[g.index.isin(sm.NEOCORTEX)]
    rows.append(dict(window=win, sub_regions=len(sub), sub_ordered=int(sub.ordered.sum()),
                     neo_regions=len(neo), neo_passive_top=int(neo.pas_top.sum())))
eo = pd.DataFrame(rows)
eo["ok"] = (eo.sub_ordered > eo.sub_regions / 2) & (eo.neo_passive_top > eo.neo_regions / 2)
print(order_tabs["a"].round(2).to_string())
print(eo.to_string(index=False))
print("Reproduced under a reading:", bool(eo.ok.any()))

**Result: reproduced.** In window a, 11 of 15 midbrain, basal ganglia and hippocampal regions show Passive < Miss < Go, and 12 of 14 neocortical regions have Passive above both Miss and Go; window b gives 12 of 15 and 12 of 14.

## fig5d-go-miss-matches-task-passive

> Regions whose pre-stimulus rate differed between task and passive contexts showed similar differences between Go and Miss trials.

Per region (Figs. 3 to 5 set, as ed6b), task minus passive (matched Go and Miss trials together, against matched passive trials) and Go minus Miss; Pearson correlation across regions.

In [ ]:
rows = []
for win in ["a", "b"]:
    g = order_tabs[win]
    tp = (g.go + g.miss) / 2 - g.pas
    gm = g.go - g.miss
    r_, p_ = ss.pearsonr(tp, gm)
    rows.append(dict(window=win, regions=len(g), r=r_, p=p_, spearman=ss.spearmanr(tp, gm)[0]))
f5d = pd.DataFrame(rows)
f5d["ok"] = (f5d.r > 0) & (f5d.p < 0.05)
print(f5d.round(5).to_string(index=False))
print("Reproduced under a reading:", bool(f5d.ok.any()))

**Result: reproduced.** Across 42 regions the task minus passive difference correlates with the Go minus Miss difference, r = 0.641 (p < 1e-5) in window a and 0.622 (p = 1e-5) in window b.

## ed7f-detect-probability-engagement

> The pre-stimulus detect probability of each area correlated with its task minus passive difference (r = 0.48, p = 0.001), as the Go minus Miss difference does.

Per neuron (Figs. 3 to 5 set) the pre-stimulus detect probability (ccCP algorithm, Go against NoGo trials T within stimulus conditions) minus 0.5; per area the mean; Pearson r with the area's task minus passive difference of fig5d. Windows (a), (b).

In [ ]:
rows = []
for win in ["a", "b"]:
    dp = stack(lambda r, w=win: None if r["dp"]["pre_" + w] is None else r["dp"]["pre_" + w][0], 1)[:, 0]
    d = pd.DataFrame(dict(region=units.region, dp=dp - 0.5))[units.fig35.to_numpy()]
    a = d.groupby("region").dp.mean()
    g = order_tabs[win]
    tp = ((g.go + g.miss) / 2 - g.pas).reindex(a.index)
    ok = tp.notna() & a.notna()
    r_, p_ = ss.pearsonr(a[ok], tp[ok])
    rows.append(dict(window=win, areas=int(ok.sum()), r=r_, p=p_, mean_dp=a.mean(),
                     dp_range=f"{a.min():.3f} to {a.max():.3f}"))
e7f = pd.DataFrame(rows)
e7f["ok"] = ((e7f.r - 0.48).abs() <= 0.048) & (e7f.p < 0.05)
print(e7f.round(4).to_string(index=False))
print("Reproduced under a reading:", bool(e7f.ok.any()))

**Result: reproduced.** The pre-stimulus detect probability of each area correlates with its task minus passive difference, r = 0.515 (p = 0.0005) in window a and 0.488 (p = 0.0011) in window b, against r = 0.48 ± 0.048 and p = 0.001.

## fig5f-engagement-index

> An engagement index, the projection of pre-stimulus population activity onto the task minus passive difference vector, was higher on Go than on Miss trials consistently across recordings (mean difference 8.42 a.u., paired t-test p < 1e-4).

Neurons N of the regions that differ in fig5c-prestimulus-task-vs-passive under the Methods' reading (all passive events, factors condition, session and neuron); the other readings' region sets are shown as variants. x_n is the neuron's mean rate -0.2 to 0 s before stimulus onset on task trials minus that on all passive trials, normalized to unit L2 norm per session; the index of a trial is x times its pre-stimulus rate vector. Go and Miss trials: Left or Right, and NoGo, of definition T, the NoGo trials having a contralateral stimulus when the contralateral contrast is above zero. Per contralateral contrast level, N trials of each, N the smaller of the Go and NoGo counts (Methods). Readings of the task trials for x: (a) trials T, (b) all task trials; of the contrast levels: (a) contralateral contrast > 0, (b) all four levels. Sessions are those kept for definition M (at least 10 matched trials per group).

In [ ]:
def eng_regions(pas_read="a", fac="a"):
    d = an[an.passive == pas_read]
    return set(d.region[d[f"p_{fac}"] < 0.0012])

def engagement(si, neuron_mask, task_read, con_read, rng):
    """Engagement index per task trial of session si, and the matched Go and Miss trials."""
    r = R[si]
    ix = uidx[si]
    m = neuron_mask[ix]
    if m.sum() == 0:
        return None
    tr = r["trials"]
    pre_t = r["pre"]["task_a"][m]
    pre_p = r["pre"]["pas_a"][m]
    tsel = tr.t_choice.isin(["L", "R", "N"]).to_numpy() if task_read == "a" else np.ones(len(tr), bool)
    x = pre_t[:, tsel].mean(axis=1) - pre_p.mean(axis=1)
    x = x / np.linalg.norm(x)
    ei = x @ pre_t
    tc = tr.t_choice.to_numpy()
    cr = tr.cr.to_numpy()
    levels = [0.25, 0.5, 1.0] if con_read == "a" else [0.0, 0.25, 0.5, 1.0]
    go_i, ms_i = [], []
    for c in levels:
        g = np.where(np.isin(tc, ["L", "R"]) & np.isclose(cr, c))[0]
        n_ = np.where((tc == "N") & np.isclose(cr, c))[0]
        k = min(len(g), len(n_))
        if k:
            go_i.extend(rng.choice(g, k, replace=False))
            ms_i.extend(rng.choice(n_, k, replace=False))
    return dict(ei=ei, go=np.array(go_i, int), miss=np.array(ms_i, int), n_neurons=int(m.sum()), x=x)

def eng_table(neuron_mask, task_read="a", con_read="a", seed=50):
    rows = []
    for si, s in enumerate(sess_names):
        if s not in keep_m:
            continue
        e = engagement(si, neuron_mask, task_read, con_read, np.random.default_rng([seed, si]))
        if e is None or len(e["go"]) < 2:
            continue
        rows.append(dict(session=s, subject=R[si]["meta"]["subject"], n_neurons=e["n_neurons"],
                         n_trials=len(e["go"]), go=e["ei"][e["go"]].mean(), miss=e["ei"][e["miss"]].mean()))
    t = pd.DataFrame(rows)
    t["diff"] = t.go - t.miss
    return t

regs_main = eng_regions("a", "a")
mask_main = (units.N & units.region.isin(regs_main)).to_numpy()
print("Regions in the index:", sorted(regs_main))

In [ ]:
rows = []
for task_read in ["a", "b"]:
    for con_read in ["a", "b"]:
        t = eng_table(mask_main, task_read, con_read)
        tt = ss.ttest_rel(t.go, t.miss)
        rows.append(dict(task=task_read, contrasts=con_read, sessions=len(t), mean_diff=t["diff"].mean(),
                         sem=t["diff"].sem(), t=tt.statistic, p=tt.pvalue, sessions_go_higher=(t["diff"] > 0).sum(),
                         neurons_per_session=t.n_neurons.mean(), neurons_se=t.n_neurons.sem()))
        if task_read == "a" and con_read == "a":
            main_t = t
for pas_read, fac in [("a", "b"), ("b", "a"), ("b", "b")]:
    mk = (units.N & units.region.isin(eng_regions(pas_read, fac))).to_numpy()
    t = eng_table(mk)
    tt = ss.ttest_rel(t.go, t.miss)
    rows.append(dict(task="a", contrasts="a", regions_from=f"fig5c passive {pas_read}, factors {fac}",
                     sessions=len(t), mean_diff=t["diff"].mean(), sem=t["diff"].sem(), t=tt.statistic,
                     p=tt.pvalue, sessions_go_higher=(t["diff"] > 0).sum(),
                     neurons_per_session=t.n_neurons.mean(), neurons_se=t.n_neurons.sem()))
f5f = pd.DataFrame(rows)
f5f["test_ok"] = (f5f.p < 1e-4) & (f5f.mean_diff > 0)
f5f["mean_ok"] = (f5f.mean_diff - 8.42).abs() <= 0.84
f5f["neur_ok"] = (f5f.neurons_per_session - 319).abs() <= 32.5
print(f5f.round(5).to_string(index=False))
print("\nPer session (readings a, a):")
print(main_t.round(2).to_string(index=False))
print("Go minus Miss by subject (mean over sessions):", main_t.groupby("subject")["diff"].mean().round(2).to_dict())

**Result: reproduced.** With trials T and contralateral contrasts above zero, the index is higher on Go than on Miss trials by 7.85 a.u. on average (s.e. 1.54; 29 of 36 sessions), paired t-test p = 1e-5, with 344 ± 34 neurons per session; expected 8.42 ± 0.84 and 319 ± 32.5. All task trials give 8.06. Including zero contrast (reading b) gives 4.69 (p = 2e-4). The sessions number 36, not 34, because the matched-contrast selection keeps 36.

## fig5e-engagement-predicts-go

> The probability of a Go response rises with the engagement index on single trials.

The index of fig5f (readings a, a), z-scored (a) within session or (b) over all trials; matched Go and Miss trials pooled over sessions; logistic regression of Go on the index (statsmodels Logit).

In [ ]:
pool = []
for si, s in enumerate(sess_names):
    if s not in keep_m:
        continue
    e = engagement(si, mask_main, "a", "a", np.random.default_rng([50, si]))
    if e is None or len(e["go"]) < 2:
        continue
    for lab, idx in [(1, e["go"]), (0, e["miss"])]:
        for i in idx:
            pool.append(dict(session=s, go=lab, ei=e["ei"][i]))
pool = pd.DataFrame(pool)
pool["z_a"] = pool.groupby("session").ei.transform(lambda x: (x - x.mean()) / x.std())
pool["z_b"] = (pool.ei - pool.ei.mean()) / pool.ei.std()
rows = []
for zr in ["z_a", "z_b"]:
    fit = smf.logit(f"go ~ {zr}", pool).fit(disp=0)
    lo, hi = fit.conf_int().loc[zr]
    rows.append(dict(zscore=zr, n_trials=len(pool), slope=fit.params[zr], ci_lo=lo, ci_hi=hi, p=fit.pvalues[zr]))
f5e = pd.DataFrame(rows)
print(f5e.to_string(index=False))
q = pd.qcut(pool.z_a, 5, labels=False)
print("P(Go) by quintile of the index (z within session):", pool.groupby(q).go.mean().round(3).tolist())
print("Reproduced under a reading:", bool(((f5e.slope > 0) & (f5e.ci_lo > 0)).any()))

**Result: reproduced.** On 2,434 matched trials, the logistic slope of Go on the index is 0.44 (95% interval 0.35 to 0.53) with the index z-scored within session, and 0.19 (0.11 to 0.27) z-scored over all trials. P(Go) rises from 0.335 to 0.614 across quintiles of the index.

## ed9i-engagement-beyond-arousal

> The engagement index improves the prediction of Go responses beyond inter-trial interval, previous reward, pupil area and motion energy (deviance test, p = 1.5e-8).

Binomial GLM of Go against Miss on matched trials (fig5f readings a, a): inter-trial interval (stimulus onset minus the previous trial's feedback time, a standard choice), previous reward (0 or 1), pupil area and face motion energy in the pre-stimulus window, z-scored within session, and their squares; against the same model plus the engagement index (z-scored within session). Deviance test: the drop in deviance against chi-squared. Windows (a) -0.2 to 0 s, (b) -0.25 to -0.05 s; pooling (a) one GLM over sessions, (b) one per session with deviances and degrees of freedom summed. Trials with a missing pupil value are dropped; sessions where the pupil is missing for all trials leave the analysis. The dataset does not say which 5 sessions the Methods excluded for unusable pupil video.

In [ ]:
glm_rows = []
for si, s in enumerate(sess_names):
    if s not in keep_m:
        continue
    e = engagement(si, mask_main, "a", "a", np.random.default_rng([50, si]))
    if e is None or len(e["go"]) < 2:
        continue
    tr = R[si]["trials"]
    for lab, idx in [(1, e["go"]), (0, e["miss"])]:
        for i in idx:
            glm_rows.append(dict(session=s, go=lab, ei=e["ei"][i], iti=tr.iti.iloc[i],
                                 prev=tr.prev_reward.iloc[i], pupil_a=tr.pupil_a.iloc[i],
                                 pupil_b=tr.pupil_b.iloc[i], me_a=tr.me_a.iloc[i], me_b=tr.me_b.iloc[i]))
G = pd.DataFrame(glm_rows)
zs = lambda x: (x - x.mean()) / x.std()
for c in ["ei", "pupil_a", "pupil_b", "me_a", "me_b"]:
    G[c] = G.groupby("session")[c].transform(zs)
G = G.dropna(subset=["iti", "prev"])
rows = []
for win in ["a", "b"]:
    base = f"go ~ iti + prev + pupil_{win} + I(pupil_{win}**2) + me_{win} + I(me_{win}**2)"
    d = G.dropna(subset=[f"pupil_{win}", f"me_{win}"])
    f0 = smf.glm(base, d, family=sm_api.families.Binomial()).fit()
    f1 = smf.glm(base + " + ei", d, family=sm_api.families.Binomial()).fit()
    dev = f0.deviance - f1.deviance
    rows.append(dict(window=win, pooling="a one GLM", sessions=d.session.nunique(), trials=len(d),
                     dev_drop=dev, df=1, p=ss.chi2.sf(dev, 1), ei_coef=f1.params["ei"]))
    dsum, dfs, ns = 0.0, 0, 0
    for s, ds in d.groupby("session"):
        try:
            g0 = smf.glm(base, ds, family=sm_api.families.Binomial()).fit()
            g1 = smf.glm(base + " + ei", ds, family=sm_api.families.Binomial()).fit()
            dsum += g0.deviance - g1.deviance
            dfs += 1
            ns += 1
        except Exception:
            pass
    rows.append(dict(window=win, pooling="b per session, summed", sessions=ns, trials=len(d),
                     dev_drop=dsum, df=dfs, p=ss.chi2.sf(dsum, dfs), ei_coef=np.nan))
e9i = pd.DataFrame(rows)
print(e9i.to_string(index=False))
print("Pupil missing on %.1f%% of matched trials" % (100 * G.pupil_a.isna().mean()))
print("Reproduced under a reading:", bool((e9i.p < 0.05).any()))

**Result: reproduced.** Adding the index to the GLM of inter-trial interval, previous reward, pupil and motion energy lowers the deviance by 63.1 (p = 2e-15, window a) and 64.0 (p = 1e-15, window b) in one GLM over 34 sessions, and by 169.7 over 34 degrees of freedom (p = 6e-20) with one GLM per session; the paper's p is 1.5e-8.

## ed9cf-engagement-controls

> The Go against Miss difference in engagement index persists on trials after rewards, after regressing out previous reward, and on trials after short and after long inter-trial intervals.

The index of fig5f (readings a, a) on matched Go and Miss trials: (1) trials after a rewarded trial; (2) all trials, with each previous-reward group's mean subtracted within session; (3) trials whose inter-trial interval is below the median, (4) above it; median readings (a) per session, (b) over all trials; tests (a) paired t-test across sessions, (b) two-sample t-test over pooled trials.

In [ ]:
P9 = G.copy()
P9["ei_res"] = P9.ei - P9.groupby(["session", "prev"]).ei.transform("mean")
rows = []
for med in ["a", "b"]:
    if med == "a":
        short = P9.iti < P9.groupby("session").iti.transform("median")
    else:
        short = P9.iti < P9.iti.median()
    sets = {"1 after reward": (P9.prev == 1, "ei"), "2 previous reward removed": (P9.prev.notna(), "ei_res"),
            "3 short ITI": (short, "ei"), "4 long ITI": (~short, "ei")}
    for nm, (sel, col) in sets.items():
        d = P9[sel]
        per = d.groupby(["session", "go"])[col].mean().unstack().dropna()
        tp = ss.ttest_rel(per[1], per[0])
        t2 = ss.ttest_ind(d.loc[d.go == 1, col], d.loc[d.go == 0, col])
        rows.append(dict(median=med, set=nm, sessions=len(per), trials=len(d),
                         go_minus_miss=(per[1] - per[0]).mean(), paired_p=tp.pvalue, pooled_p=t2.pvalue,
                         go_higher=(per[1] - per[0]).mean() > 0))
e9c = pd.DataFrame(rows)
print(e9c.round(6).to_string(index=False))
print("The index is z-scored within session here (as for the GLM).")
okc = {(m, t): bool(((e9c[e9c["median"] == m][f"{t}_p"] < 0.05) & e9c[e9c["median"] == m].go_higher).all())
       for m in ["a", "b"] for t in ["paired", "pooled"]}
print("All four sets Go higher with p < 0.05, by (median, test):", okc)

**Result: reproduced.** The z-scored index is higher on Go than on Miss trials after rewarded trials (0.37, paired p = 7e-5), with previous reward removed (0.34, p = 4e-5), after short (0.47, p = 7e-6) and long inter-trial intervals (0.32, p = 0.005), with the median taken per session; the median over all trials and the pooled two-sample test give p below 0.002 in every set.

## ed9k-engagement-vector-specific

> The engagement vector correlates better with the Go minus Miss vector than with the vectors of other behavioural variables, though not significantly for previous reward (p = 0.096).

Per recording, the Pearson correlation over neurons (those of the index) between the engagement vector x and the Go minus Miss vector (mean pre-stimulus rate on matched Go minus Miss trials), and between x and each variable's vector over task trials T: (a) the difference of mean pre-stimulus rates between trials above and below the variable's median (previous reward, rewarded minus not); (b) each neuron's regression slope on the variable. Tests across recordings (a) paired t-test, (b) Wilcoxon signed-rank.

In [ ]:
rows = []
for si, s in enumerate(sess_names):
    if s not in keep_m:
        continue
    e = engagement(si, mask_main, "a", "a", np.random.default_rng([50, si]))
    if e is None or len(e["go"]) < 2 or e["n_neurons"] < 3:
        continue
    r = R[si]
    m = mask_main[uidx[si]]
    pre_t = r["pre"]["task_a"][m]
    tr = r["trials"]
    gm = pre_t[:, e["go"]].mean(axis=1) - pre_t[:, e["miss"]].mean(axis=1)
    row = dict(session=s, r_gomiss=ss.pearsonr(e["x"], gm)[0])
    T_ = tr.t_choice.isin(["L", "R", "N"]).to_numpy()
    for v, col in [("pupil", "pupil_a"), ("me", "me_a"), ("iti", "iti"), ("prev", "prev_reward")]:
        val = tr[col].to_numpy()
        ok = T_ & np.isfinite(val)
        if ok.sum() < 10 or np.nanstd(val[ok]) == 0:
            row[f"{v}_a"] = row[f"{v}_b"] = np.nan
            continue
        hi = val[ok] > (np.median(val[ok]) if v != "prev" else 0.5)
        va = pre_t[:, ok][:, hi].mean(axis=1) - pre_t[:, ok][:, ~hi].mean(axis=1)
        xv = val[ok] - val[ok].mean()
        vb = (pre_t[:, ok] - pre_t[:, ok].mean(axis=1, keepdims=True)) @ xv / (xv @ xv)
        row[f"{v}_a"] = ss.pearsonr(e["x"], va)[0] if np.std(va) > 0 else np.nan
        row[f"{v}_b"] = ss.pearsonr(e["x"], vb)[0] if np.std(vb) > 0 else np.nan
    rows.append(row)
K = pd.DataFrame(rows)
out = []
for rd in ["a", "b"]:
    for v in ["pupil", "me", "iti", "prev"]:
        d = K[["r_gomiss", f"{v}_{rd}"]].dropna()
        out.append(dict(vector=rd, variable=v, recordings=len(d), r_gomiss=d.r_gomiss.mean(),
                        r_var=d[f"{v}_{rd}"].mean(), ttest_p=ss.ttest_rel(d.r_gomiss, d[f"{v}_{rd}"]).pvalue,
                        wilcoxon_p=ss.wilcoxon(d.r_gomiss, d[f"{v}_{rd}"]).pvalue))
e9k = pd.DataFrame(out)
print(e9k.round(5).to_string(index=False))
for rd in ["a", "b"]:
    for test in ["ttest_p", "wilcoxon_p"]:
        d = e9k[e9k.vector == rd]
        ok = bool((d.r_gomiss > d.r_var).all() and (d[d.variable != "prev"][test] < 0.05).all())
        print(f"vector {rd}, {test}: Go-Miss larger for all four and p < 0.05 for pupil, motion energy, ITI:", ok)

**Result: reproduced.** The engagement vector correlates with the Go minus Miss vector at r = 0.21 on average, more than with the pupil (r = -0.07, paired p = 0.0002), motion energy (0.05, p = 0.004), inter-trial interval (0.02, p = 0.0005) and previous reward vectors (0.14, p = 0.13; Wilcoxon p = 0.077), with median-split vectors; regression slopes give the same pattern. As in the paper, the previous reward comparison is not significant.

## suppfig3-nonresponsive-neurons

> The engagement index differs between Go and Miss trials even when computed from neurons that are otherwise not responsive in the task.

As fig5f (readings a, a), from all neurons of the index regions, from those of the Figs. 3 to 5 set, and from the others.

In [ ]:
rows = []
for nm, mk in [("all", mask_main), ("responsive (Figs. 3-5 set)", mask_main & units.fig35.to_numpy()),
               ("others", mask_main & ~units.fig35.to_numpy())]:
    t = eng_table(mk)
    tt = ss.ttest_rel(t.go, t.miss)
    rows.append(dict(neurons=nm, sessions=len(t), neurons_per_session=t.n_neurons.mean(),
                     mean_diff=t["diff"].mean(), t=tt.statistic, p=tt.pvalue))
s3 = pd.DataFrame(rows)
s3["ok"] = (s3.p < 0.05) & (s3.mean_diff > 0)
print(s3.to_string(index=False))
print("Reproduced:", bool(s3.ok.all()))

**Result: reproduced.** Go is higher than Miss with the index from all neurons (7.85, p = 1e-5), from responsive neurons (7.24, p = 4e-5) and from the other neurons alone (3.07, p = 9e-4), over 36 sessions.

## ed7b-detect-probability-widespread

> Neurons with significant detect probability (Go against NoGo for the same stimulus) around movement onset are found widely across regions, replicating the global action signal.

ccCP algorithm (Methods; code ccCP/choiceProbShuf.m) on neurons N: Go (Left or Right) against NoGo trials T within each of the 16 stimulus conditions, 2,000 within-condition label shuffles; significant when the observed value lies below 2.5% or above 97.5% of the shuffles. NoGo trials take a movement time drawn from the session's Go trials. Windows (a) -0.1 to 0.05 s and (b) -0.05 to 0 s from movement onset. Sessions with fewer than 10 includable trials (in conditions with both Go and NoGo) are left out. A region is marked when the Clopper-Pearson 95% interval of its fraction lies above 5%.

In [ ]:
from statsmodels.stats.proportion import proportion_confint
rows = []
for w in ["a", "b"]:
    frac = stack(lambda r, w=w: None if r["dp"][w] is None else r["dp"][w][1], 1)[:, 0]
    ninc = pd.Series({r["meta"]["name"]: (0 if r["dp"][w] is None else r["dp"][w][2]) for r in R})
    keep = set(ninc.index[ninc >= 10])
    d = pd.DataFrame(dict(region=units.region, session=units.session, f=frac))
    d = d[units.N.to_numpy() & d.session.isin(keep) & np.isfinite(d.f)]
    d["sig"] = (d.f < 0.025) | (d.f > 0.975)
    g = d.groupby("region").sig.agg(["sum", "size"])
    lo, hi = proportion_confint(g["sum"], g["size"], alpha=0.05, method="beta")
    g["pct"] = 100 * g["sum"] / g["size"]
    g["ci_lo"] = 100 * lo
    g["marked"] = lo > 0.05
    rows.append(dict(window=w, sessions_kept=len(keep), sessions_left_out=39 - len(keep),
                     neurons=len(d), pct_sig=100 * d.sig.mean(), regions=len(g),
                     regions_marked=int(g.marked.sum())))
    if w == "a":
        print(g.round(1).T.to_string())
        print("Regions not marked:", g.index[~g.marked].tolist())
e7b = pd.DataFrame(rows)
e7b["ok"] = e7b.regions_marked >= 0.75 * e7b.regions
print(e7b.round(2).to_string(index=False))
print("Reproduced under a reading:", bool(e7b.ok.any()))

**Result: reproduced.** Around movement onset 28.18% of neurons N have a significant detect probability in window a, and all 42 regions are marked above the 5% chance level (40 of 42 in window b, 17.38%). No session has fewer than 10 includable trials.

## jPECC (fig4d claims)

jPECC as in the Methods and code jPECC.m: per recording, populations VIS, Frontal (MOs, MOp, PL) and Midbrain (MRN, SCm, SNr, ZI), each of the regions with at least 13 neurons in the session; spike counts in 10 ms bins, causal half-Gaussian of s.d. 25 ms, divided by each neuron's baseline + 1 sp/s; per group one PCA across time points and trials to min(10, neurons of either group) dimensions, fitted on the trials and bins of the window and alignment analysed (Methods, jPECC analysis: "For each region group, principal components analysis was performed across time points and trials to reduce population activity to 10 dimensions"); 10-fold cross-validation over trials; CCA on training PCs with ridge rows (lambda 0.5) appended; Pearson r of the test projections on the first canonical pair, for every pair of time bins. Stimulus-aligned window -0.05 to 0.3 s, movement-aligned -0.3 to 0.05 s (35 bins each; a standard choice that covers the times the claims name). Asymmetry index at t: the jPECC matrix sliced diagonally from -50 to +50 ms around t (Methods, jPECC analysis: "the average along a vector from [t-50, t+50] to [t,t] ... subtracted from ... [t, t] to [t+50, t-50]"), the half where the first population is earlier minus the half where it is later, so that positive means the first leads; t-test across recordings against 0 at each t, alpha 0.05. Readings: baseline (a) -0.2 to 0 s, (b) -0.1 to 0 s before stimulus onset; trials for the stimulus alignment (a) all trials T, (b) Left and Right choice trials (movement alignment always uses Left and Right trials). The criterion was amended after a question (`questions/steinmetz-2019.yaml`): the PCA across time points and trials of the Methods is now the criterion's method, and the PCA per time bin of jPECC.m, the method of the first run, is a variant. Both are run under every reading.

In [ ]:
jt0 = time.time()
JB = 0.01
def jp_norm(arr, base):
    rate = arr.astype(float) / JB
    rate = sm.causal_smooth(rate, sm.JP_SMOOTH, axis=2, bin_size=JB)
    return rate / (base[:, None, None] + 1.0)

jp_rows = []
pairs = [("VIS", "Frontal"), ("VIS", "Midbrain"), ("Frontal", "Midbrain")]
for si, r in enumerate(R):
    jp = r["jpecc"]
    tc = r["trials"].t_choice.to_numpy()
    Tm = np.isin(tc, ["L", "R", "N"])
    Gm = np.isin(tc, ["L", "R"])
    for g1, g2 in pairs:
        if g1 not in jp or g2 not in jp:
            continue
        for bread, (blo, bhi) in {"a": (-0.2, 0.0), "b": (-0.1, 0.0)}.items():
            arrs = {}
            for g in (g1, g2):
                st = jp[g]["stim"]
                bi = slice(int(round((blo + 0.3) / JB)), int(round((bhi + 0.3) / JB)))
                base = st[:, Tm, bi].mean(axis=(1, 2)) / JB
                arrs[g] = (jp_norm(st, base)[:, :, 25:60], jp_norm(jp[g]["move"], base)[:, :, 0:35])
            combos = [(al, tr_, ts, pca) for pca in ("all", "bin")
                      for al, tr_, ts in [("stim", "a", Tm), ("stim", "b", Gm), ("move", "-", Gm)]]
            for align, tread, tsel, pca in combos:
                k = 0 if align == "stim" else 1
                a1 = np.transpose(arrs[g1][k][:, tsel], (1, 2, 0))
                a2 = np.transpose(arrs[g2][k][:, tsel], (1, 2, 0))
                J = sm.jpecc(a1, a2, np.random.default_rng([60, si]), pca=pca)
                jp_rows.append(dict(session=r["meta"]["name"], pair=f"{g1}-{g2}", baseline=bread, pca=pca,
                                    trials=tread, align=align, asym=sm.asymmetry(J),
                                    n1=a1.shape[2], n2=a2.shape[2], ntr=a1.shape[0]))
JPD = pd.DataFrame(jp_rows)
print("jPECC computed in %.0f s; recordings per pair:" % (time.time() - jt0),
      JPD.groupby("pair").session.nunique().to_dict())
t_stim = -0.05 + JB * np.arange(35)
t_move = -0.3 + JB * np.arange(35)

def asym_test(pair, align, baseline, trials, pca="all"):
    d = JPD[(JPD["pair"] == pair) & (JPD["align"] == align) & (JPD["baseline"] == baseline)
            & (JPD["trials"] == trials) & (JPD["pca"] == pca)]
    A = np.vstack(d.asym.to_numpy())
    tt = ss.ttest_1samp(A, 0, axis=0, nan_policy="omit")
    return pd.DataFrame(dict(t=(t_stim if align == "stim" else t_move), mean=np.nanmean(A, axis=0),
                             tstat=tt.statistic, p=tt.pvalue, n=len(d)))

## fig4d-jpecc-vis-leads

> In jPECC, visual cortex activity leads frontal cortex activity by about 40 ms in the period about 200 ms before movement, and visual cortex leads frontal cortex and midbrain at about 100 ms after stimulus onset.

Reading of the criterion's open points (they are open in the claims file's checks): "VIS leads Frontal at time points between -0.2 and -0.05 s" holds when at least one time point in that range has a positive asymmetry with p < 0.05; "near 0.1 s" is the points from 0.08 to 0.12 s, at least one with positive asymmetry and p < 0.05 for each of VIS-Frontal and VIS-Midbrain.

In [ ]:
rows = []
for pca, bread, tread in [(p_, b_, t_) for p_ in ("all", "bin") for b_ in ("a", "b") for t_ in ("a", "b")]:
    mv = asym_test("VIS-Frontal", "move", bread, "-", pca)
    w = mv[(mv.t >= -0.2 - 1e-9) & (mv.t <= -0.05 + 1e-9)]
    mv_ok = ((w.p < 0.05) & (w["mean"] > 0)).sum()
    res_ = dict(baseline=bread, trials=tread, pca=pca, n_vis_front=int(mv.n.iloc[0]),
                move_sig_pos_points=int(mv_ok), move_points=len(w),
                move_mean_asym=w["mean"].mean())
    for pr in ["VIS-Frontal", "VIS-Midbrain"]:
        st = asym_test(pr, "stim", bread, tread, pca)
        w2 = st[(st.t >= 0.08 - 1e-9) & (st.t <= 0.12 + 1e-9)]
        res_[f"{pr}_stim_sig_pos"] = int(((w2.p < 0.05) & (w2["mean"] > 0)).sum())
        res_[f"{pr}_stim_mean"] = w2["mean"].mean()
        res_[f"{pr}_n"] = int(st.n.iloc[0])
    rows.append(res_)
f4d = pd.DataFrame(rows)
f4d["ok"] = (f4d.move_sig_pos_points > 0) & (f4d["VIS-Frontal_stim_sig_pos"] > 0) & (f4d["VIS-Midbrain_stim_sig_pos"] > 0)
print("pca: all = one PCA per group across time points and trials (criterion); bin = per time bin (variant)")
print(f4d.round(4).to_string(index=False))
print("\nPCA across time points and trials. VIS-Frontal, movement aligned, baseline a:")
print(asym_test("VIS-Frontal", "move", "a", "-").round(4).to_string(index=False))
print("\nVIS-Frontal and VIS-Midbrain, stimulus aligned, baseline a, trials a:")
print(asym_test("VIS-Frontal", "stim", "a", "a").merge(asym_test("VIS-Midbrain", "stim", "a", "a"),
      on="t", suffixes=("_VF", "_VM")).round(4).to_string(index=False))
for bread in ["a", "b"]:
    st = asym_test("VIS-Midbrain", "stim", bread, "b")
    print("\nPCA across time points and trials. VIS-Midbrain, stimulus aligned, baseline %s, trials b, 0.06 to 0.14 s:" % bread)
    print(st[(st.t >= 0.06 - 1e-9) & (st.t <= 0.14 + 1e-9)].round(4).to_string(index=False))

# Dependence on the random folds (a standard choice): VIS-Midbrain, stimulus aligned, trials b,
# PCA across time points and trials, with five other fold seeds.
seed_rows = []
for seed in range(1, 6):
    for bread, (blo, bhi) in {"a": (-0.2, 0.0), "b": (-0.1, 0.0)}.items():
        A = []
        for si, r in enumerate(R):
            jp = r["jpecc"]
            if "VIS" not in jp or "Midbrain" not in jp:
                continue
            tc = r["trials"].t_choice.to_numpy()
            Tm, Gm = np.isin(tc, ["L", "R", "N"]), np.isin(tc, ["L", "R"])
            bi = slice(int(round((blo + 0.3) / JB)), int(round((bhi + 0.3) / JB)))
            ar = []
            for g in ("VIS", "Midbrain"):
                st = jp[g]["stim"]
                base = st[:, Tm, bi].mean(axis=(1, 2)) / JB
                ar.append(np.transpose(jp_norm(st, base)[:, :, 25:60][:, Gm], (1, 2, 0)))
            A.append(sm.asymmetry(sm.jpecc(ar[0], ar[1], np.random.default_rng([600 + seed, si]), pca="all")))
        A = np.vstack(A)
        tt = ss.ttest_1samp(A, 0, axis=0, nan_policy="omit")
        w = (t_stim >= 0.08 - 1e-9) & (t_stim <= 0.12 + 1e-9)
        seed_rows.append(dict(seed=seed, baseline=bread, recordings=len(A),
                              sig_pos_points=int(((tt.pvalue[w] < 0.05) & (np.nanmean(A, axis=0)[w] > 0)).sum()),
                              min_p=np.nanmin(tt.pvalue[w]), mean_asym=np.nanmean(A[:, w])))
print("\nVIS-Midbrain near 0.1 s (0.08 to 0.12 s), trials b, PCA across time points, other fold seeds:")
print(pd.DataFrame(seed_rows).round(4).to_string(index=False))
print("Reproduced under a reading of the criterion (pca all):", bool(f4d[f4d.pca == "all"].ok.any()),
      "| under the per-bin variant:", bool(f4d[f4d.pca == "bin"].ok.any()))

**Result: reproduced.** With the PCA across time points and trials of the amended criterion, VIS leads Frontal before movement at 3 to 5 of 16 time points between -0.2 and -0.05 s (positive, p < 0.05; 17 recordings), and after the stimulus at 3 points from 0.08 to 0.12 s in every reading. VIS leads Midbrain near 0.1 s only with trials reading b (Left and Right choice trials), at one point, 0.08 s (mean 0.098, p = 0.003 with baseline a; 0.074, p = 0.025 with baseline b; 11 recordings); with trials a no point is significant. The criterion asks for one reading, so the claim holds, on a single time point at the edge of the window. With five other fold seeds that point stays significant under baseline a in all five, and under baseline b in three. With the per-bin PCA of jPECC.m, the method of the first run and now a variant, VIS does not lead Midbrain in any reading.

## fig4d-jpecc-no-lag-frontal-midbrain

> Population activity in midbrain and frontal cortex shows no consistent lead or lag.

A claim of absence on the paper's test: the t-test across recordings of the Frontal-Midbrain asymmetry index against 0 at each time point from -0.25 to 0 s from movement onset, alpha 0.05 uncorrected. Reproduced when no point is significant.

In [ ]:
rows = []
for pca, bread in [("all", "a"), ("all", "b"), ("bin", "a"), ("bin", "b")]:
    mv = asym_test("Frontal-Midbrain", "move", bread, "-", pca)
    w = mv[(mv.t >= -0.25 - 1e-9) & (mv.t <= 0.0 + 1e-9)]
    rows.append(dict(baseline=bread, pca=pca, recordings=int(mv.n.iloc[0]), points=len(w),
                     sig_points=int((w.p < 0.05).sum()), sig_bonferroni=int((w.p < 0.05 / len(w)).sum()),
                     mean_asym=w["mean"].mean(), min_p=w.p.min(),
                     max_abs_mean=w["mean"].abs().max()))
f4n = pd.DataFrame(rows)
print("pca: all = one PCA per group across time points and trials (criterion); bin = per time bin (variant)")
print(f4n.round(4).to_string(index=False))
print("PCA across time points and trials, baseline a:")
print(asym_test("Frontal-Midbrain", "move", "a", "-").round(4).to_string(index=False))
print("PCA across time points and trials, baseline b:")
print(asym_test("Frontal-Midbrain", "move", "b", "-").round(4).to_string(index=False))
m_ = f4n.pca == "all"
print("No significant point under every reading of the criterion:", bool((f4n[m_].sig_points == 0).all()),
      "| under any:", bool((f4n[m_].sig_points == 0).any()))

**Result: not reproduced.** On the paper's uncorrected test, with the PCA across time points and trials of the amended criterion, the Frontal-Midbrain asymmetry is significant at 1 of 26 time points from -0.25 to 0 s under each baseline (-0.12 s, Midbrain leading; mean -0.063, p = 0.028 with baseline a, -0.054, p = 0.017 with baseline b; 9 recordings). The mean index over the window is -0.012 (a) and -0.022 (b), and no point survives Bonferroni correction over 26 points. The per-bin PCA variant gives 1 and 2 significant points.

## Kernel regression claims: cost on a subset

Eleven claims rest on kernel test K (fig3c, fig3d, fig3e, ed4f, ed5h, fig4b, fig4c, fig4f, fig4fg, ed5g, ed7c). The cell below builds the design of definition K for one session (the first file), fits the full model of one neuron as the criterion gives it (elastic net, alpha 0.5, lambda 0.5, 5-fold cross-validation over trials, 1 to 50 reduced-rank components), times it, and scales the time to the analysis. scikit-learn's ElasticNet stands in for glmnet (same penalty; it is used here only for the timing). The reduced-rank basis is taken from this session's neurons alone for the timing.

In [ ]:
from sklearn.linear_model import ElasticNet
kt0 = time.time()
S0 = sm.Session(paths[0], np.random.default_rng([1, 0]))
tr0 = S0.trials
inc0 = np.where(tr0.t_choice.isin(["L", "R", "N"]).to_numpy())[0]
nb_s = S0.stim_counts.shape[2]
rate = sm.causal_smooth(S0.stim_counts[:, inc0].astype(np.float32) / sm.BIN, 0.025)
t_bins = sm.STIM_WIN[0] + sm.BIN * np.arange(nb_s)
base = rate[:, :, (t_bins >= -0.2) & (t_bins < 0)].mean(axis=(1, 2))
rows_X, rows_Y, trial_id = [], [], []
ucl = [0.25, 0.5, 1.0]
for j, i in enumerate(inc0):
    mt = tr0.first_move.iloc[i] if tr0.t_choice.iloc[i] in ("L", "R") else np.nan
    t_end = min(0.4, mt + 0.025) if np.isfinite(mt) else 0.4
    samp = np.where((t_bins >= -0.05) & (t_bins < t_end))[0]
    X = np.zeros((len(samp), 6 * 90 + 2 * 55))
    lag_v = np.round((t_bins[samp] + 0.05) / sm.BIN).astype(int)    # 0..89 from -0.05 s
    for side, c in [(0, tr0.cl.iloc[i]), (1, tr0.cr.iloc[i])]:
        if c > 0:
            k = side * 3 + ucl.index(c)
            ok = (lag_v >= 0) & (lag_v < 90)
            X[np.where(ok)[0], k * 90 + lag_v[ok]] = 1
    if np.isfinite(mt):
        lag_m = np.round((t_bins[samp] - mt + 0.25) / sm.BIN).astype(int)   # 0..54 from -0.25 s
        ok = (lag_m >= 0) & (lag_m < 55)
        X[np.where(ok)[0], 540 + lag_m[ok]] = 1
        X[np.where(ok)[0], 595 + lag_m[ok]] = 1 if tr0.t_choice.iloc[i] == "L" else -1
    rows_X.append(X)
    rows_Y.append(rate[:, j, samp].T - base[None, :])
    trial_id.append(np.full(len(samp), j))
X = np.vstack(rows_X); Y = np.vstack(rows_Y); trial_id = np.concatenate(trial_id)
X1 = np.hstack([X, np.ones((len(X), 1))])
B = np.linalg.lstsq(X1, Y, rcond=None)[0]
_, _, vt = np.linalg.svd(X1 @ B, full_matrices=False)
Wb = np.linalg.lstsq(X1, X1 @ B @ vt[:200].T, rcond=None)[0]     # predictors -> 200 components
Zc = X1 @ Wb
print("Design: %d samples, %d predictors, %d neurons; built in %.1f s" % (X.shape[0], X.shape[1], Y.shape[1], time.time() - kt0))
folds = np.array_split(np.random.default_rng(70).permutation(len(inc0)), 5)
y = Y[:, int(np.argmax(Y.var(axis=0)))]
t_fit = time.time()
n_fits = 0
for k in range(1, 51):
    for f in folds:
        te = np.isin(trial_id, f)
        ElasticNet(alpha=0.5, l1_ratio=0.5, max_iter=1000).fit(Zc[~te, :k], y[~te])
        n_fits += 1
per_neuron = time.time() - t_fit
n_full = int(units.fig35.sum())
fits_nested = 3 * (18 * 10 + 1)       # Vision, Action, Choice: 1 to 18 components x 10 folds, plus the 75-component fit
est_full = per_neuron * n_full
est_nested = per_neuron / n_fits * fits_nested * 4240   # about 4,240 neurons pass VE > 2% (claims, ed5h-choice-shuffle-fpr)
est_total = est_full + est_nested
print("One neuron, full model: %d fits in %.1f s" % (n_fits, per_neuron))
print("Neurons entering the full model (Figs. 3-5 set): %d -> %.1f h on one core" % (n_full, est_full / 3600))
print("Nested tests (3 kernels, about 4,240 neurons): %.1f h; total %.1f h on one core, %.1f h on 4 cores"
      % (est_nested / 3600, est_total / 3600, est_total / 3600 / 4))
print("The Choice shuffle of ed5h repeats the whole test once more. Notebook limit: 1 h.")
del S0, rate, X, Y, X1, Zc

In [ ]:
print(f"Notebook run time {time.time() - T0:.0f} s; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1c-easy-performance | reproduced |
| fig1c-low-contrast-performance | reproduced |
| fig1c-competing-performance | reproduced |
| fig1d-reaction-time-difficulty | reproduced |
| ed1q-wheel-choice-decoding | reproduced |
| early-response-fraction | reproduced |
| passive-no-movement | not reproduced |
| recording-counts | reproduced |
| edtable1-region-counts | reproduced |
| supptable2-sessions | reproduced (assumption: finer region labels mapped to their parent) |
| task-modulated-fraction | reproduced |
| suppfig2-rate-amplitude | partially reproduced |
| fig2de-activity-order | reproduced |
| fig2fg-miss-restricted | partially reproduced |
| fig5b-miss-more-than-passive | partially reproduced |
| fig5c-prestimulus-task-vs-passive | reproduced |
| ed6b-prestimulus-order | reproduced |
| fig5d-go-miss-matches-task-passive | reproduced |
| ed7f-detect-probability-engagement | reproduced |
| fig5f-engagement-index | reproduced |
| fig5e-engagement-predicts-go | reproduced |
| ed9i-engagement-beyond-arousal | reproduced |
| ed9cf-engagement-controls | reproduced |
| ed9k-engagement-vector-specific | reproduced |
| suppfig3-nonresponsive-neurons | reproduced |
| ed7b-detect-probability-widespread | reproduced |
| fig4d-jpecc-vis-leads | reproduced |
| fig4d-jpecc-no-lag-frontal-midbrain | not reproduced |
| fig3c, fig3d, fig3e, ed4f, ed5h, fig4b, fig4c, fig4f, fig4fg, ed5g, ed7c (kernel regression) | blocked: too large |
| histology-alignment-error | blocked: data missing |

**How the analysis differs from the paper's.** Trials follow definition T with the dataset's typed wheel movements (types 1 and 2 are the turns that would register a choice) and its `included` flag for engagement, which agrees with the Methods' rule on 98.3% of trials. Regions come from each unit's peak channel; finer Allen labels (SCig, SCsg, LSr, LSc, ORBm, COA, TT, PIR, DP) are mapped to their parent among the 42 regions, an assumption. Rates are spike counts in the windows of the Methods. The z-score of definition R is infinite for neurons without baseline spikes; both treatments are run. jPECC follows the criterion as amended after a question (one PCA per group across time points and trials, as the Methods say) and the Methods' diagonal slice for the asymmetry index; the per-bin PCA of jPECC.m is a variant. Extended Data Table 1 and the areas of Supplementary Table 2 count every neuron of the 42 regions, as the criteria amended after a question say. The engagement index uses the regions of the Methods' ANOVA reading. The kernel regression claims were not run: the full model and nested tests are estimated above at several hours.

**What was found about the dandiset.**

- The trial table, wheel movements and spikes reproduce the paper's behavioural numbers closely: 9,538 engaged trials (the paper's count), and the Left and Right trials of definition T (1,970 and 1,729) match the code's inclusion arrays (1,971 and 1,729).
- The `rate` attribute of `wheel_position` (and of `lickPiezo` and `face_motion_energy`) holds the sampling interval in seconds (0.0004 s, 2.5 kHz), not a rate in Hz; the sample counts times this value give the recording lengths.
- `peak_channel` is 1-based: the row `peak_channel - 1` of the electrodes table matches the unit's depth better than the row `peak_channel`. This is not documented.
- `wheel_moves` is an IntervalSeries whose `data` (movement type) has half the length of its `timestamps` (interleaved onsets and offsets).
- Some units carry finer Allen labels than the paper's 42 regions (2,262 units), which must be mapped to their parent to recover the paper's counts.
- Genotypes are written differently from Supplementary Table 1 (Ai95 x VGlut1-Cre for RCL-GCaMP6f x VGlut-Cre; Vglut1-Cre for Vglut1-IRES2-Cre-D); sexes and ages match.
- Supplementary Table 2's areas match the regions of each session's neurons in 38 of 39 sessions; Extended Data Table 1's recordings, mice and neurons match in 38 of 42 regions when every neuron of the 42 regions is counted (22,434 against 22,458 neurons).
- The dandiset does not say which 5 sessions had unusable pupil video; two files have missing values in `eye_area`.
- The dandiset holds no histology (DiI spot coordinates or landmarks).
- In the passive replay 84.6% of high-contrast presentations have no detected movement, against 94.1% in the paper.